# 110 — LINCS/CMap Acquisition and Audit

## Objective

Audit and freeze the LINCS/CMap perturbational resources required for
Phase 7 — Perturbational Hypotheses.

The primary perturbational source is the Expanded CMap LINCS 2020 release,
with the Level 5 small-molecule (`trt_cp`) matrix as the primary expression
resource and the associated signature, gene, compound, cell-line, instance,
and release metadata as supporting inputs.

This notebook will establish file identity and source provenance, verify the
structural integrity and internal consistency of the acquired resources, and
perform an outcome-blind feasibility audit required before the Phase 7
analytical specification is finalized.

## Main tasks

This notebook will:

1. verify the presence, size, and SHA-256 identity of all acquired LINCS/CMap
   files;
2. document source, release, acquisition role, and provenance for each file;
3. verify that the Level 5 `trt_cp` GCTX resource is readable and structurally
   consistent with its documented dimensions;
4. audit the schemas and identifier systems of `siginfo`, `geneinfo`,
   `compoundinfo`, `cellinfo`, and `instinfo`;
5. verify referential consistency across signature, perturbagen, cell-line,
   instance, and gene identifiers;
6. characterize the available L1000 gene spaces, including landmark/measured
   and inferred genes;
7. characterize, without inspecting connectivity results, the available
   compound × cell-line × dose × time coverage and the distribution of unique
   cell-line and lineage support;
8. quantify outcome-blind representation coverage of the three frozen Phase 4
   consensus programs in the available L1000 gene spaces; and
9. produce the audited raw-data handoff required before notebooks 700–703.

## Analytical boundary

This notebook is an acquisition, integrity, metadata, and feasibility audit.

It will not:

- calculate program–perturbation connectivity scores;
- inspect inverse-signature results;
- prioritize compounds or mechanisms of action;
- select doses, times, cell lines, or compounds based on favorable
  perturbational results;
- use Phase 5 or Phase 6 results to redefine the Phase 7 analytical universe;
- redefine, reorient, reweight, rescue, or exclude the frozen Phase 4
  consensus programs;
- determine therapeutic efficacy or biological reversal; or
- perform the final cross-evidence therapeutic prioritization reserved for
  Phase 9.

Any eligibility thresholds or analytical choices informed by this notebook
must be based only on technical metadata, coverage, integrity, or
representation feasibility and must be frozen before connectivity results are
inspected.

## Downstream handoff

Notebook 110 will provide the audited LINCS/CMap resource definition and the
outcome-blind feasibility information required to finalize the Phase 7
analysis contract.

Only after that specification is frozen will notebook 700 construct the
formal program-query signatures and notebook 701 perform connectivity
analysis.

In [ ]:
# =============================================================================
# Imports
# =============================================================================

import json

import h5py
import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path
from pancancer_epigenetics.utils.raw_data_registry import (
    load_raw_data_registry,
    validate_raw_data_registry,
)

In [ ]:
# =============================================================================
# LINCS/CMap input paths
# =============================================================================

LINCS_CELLINFO_PATH = (
    Paths.lincs
    / "cellinfo_beta.txt"
)

LINCS_COMPOUNDINFO_PATH = (
    Paths.lincs
    / "compoundinfo_beta.txt"
)

LINCS_GENEINFO_PATH = (
    Paths.lincs
    / "geneinfo_beta.txt"
)

LINCS_INSTINFO_PATH = (
    Paths.lincs
    / "instinfo_beta.txt"
)

LINCS_LEVEL5_TRT_CP_PATH = (
    Paths.lincs
    / "level5_beta_trt_cp_n720216x12328.gctx"
)

LINCS_METADATA_DEFINITIONS_PATH = (
    Paths.lincs
    / "LINCS2020 Release Metadata Field Definitions.xlsx"
)

LINCS_README_PATH = (
    Paths.lincs
    / "README.txt"
)

LINCS_SIGINFO_PATH = (
    Paths.lincs
    / "siginfo_beta.txt"
)

In [ ]:
# =============================================================================
# Validate acquired LINCS/CMap inputs
# =============================================================================

lincs_input_paths = {
    "cellinfo": LINCS_CELLINFO_PATH,
    "compoundinfo": LINCS_COMPOUNDINFO_PATH,
    "geneinfo": LINCS_GENEINFO_PATH,
    "instinfo": LINCS_INSTINFO_PATH,
    "level5_trt_cp": LINCS_LEVEL5_TRT_CP_PATH,
    "metadata_definitions": LINCS_METADATA_DEFINITIONS_PATH,
    "readme": LINCS_README_PATH,
    "siginfo": LINCS_SIGINFO_PATH,
}

missing_inputs = [
    name
    for name, path in lincs_input_paths.items()
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing acquired LINCS/CMap inputs: "
        + ", ".join(missing_inputs)
    )

for name, path in lincs_input_paths.items():
    print(f"{name}: {project_relative_path(path)}")

In [ ]:
# =============================================================================
# Byte-level inventory of acquired LINCS/CMap files
# =============================================================================

lincs_file_inventory = pd.DataFrame(
    [
        {
            "resource": resource,
            "relative_path": project_relative_path(path),
            "size_bytes": path.stat().st_size,
            "sha256": calculate_sha256(path),
        }
        for resource, path in lincs_input_paths.items()
    ]
)

lincs_file_inventory

In [ ]:
# =============================================================================
# Report complete byte-level identities
# =============================================================================

for row in lincs_file_inventory.itertuples(index=False):
    print(f"{row.resource}")
    print(f"  path:   {row.relative_path}")
    print(f"  bytes:  {row.size_bytes:,}")
    print(f"  sha256: {row.sha256}")

In [ ]:
# =============================================================================
# Inspect Level 5 trt_cp GCTX structure
# =============================================================================

with h5py.File(LINCS_LEVEL5_TRT_CP_PATH, mode="r") as gctx:
    print("Top-level groups:", list(gctx.keys()))

    matrix = gctx["0/DATA/0/matrix"]
    row_ids = gctx["0/META/ROW/id"]
    col_ids = gctx["0/META/COL/id"]

    print(f"Matrix shape: {matrix.shape}")
    print(f"Matrix dtype: {matrix.dtype}")
    print(f"Row IDs:      {row_ids.shape[0]:,}")
    print(f"Column IDs:   {col_ids.shape[0]:,}")

In [ ]:
# =============================================================================
# Validate Level 5 GCTX dimensional consistency
# =============================================================================

with h5py.File(LINCS_LEVEL5_TRT_CP_PATH, mode="r") as gctx:
    matrix = gctx["0/DATA/0/matrix"]
    row_ids = gctx["0/META/ROW/id"]
    col_ids = gctx["0/META/COL/id"]

    n_genes = row_ids.shape[0]
    n_signatures = col_ids.shape[0]

    # GCTX stores the numerical dataset transposed relative to the logical
    # row × column representation exposed by cmapPy.
    assert matrix.shape == (n_signatures, n_genes)
    assert n_genes == 12328
    assert n_signatures == 720216

print("Level 5 trt_cp GCTX dimensional consistency: PASS")
print(f"Logical matrix: {n_genes:,} genes × {n_signatures:,} signatures")

In [ ]:
# =============================================================================
# Inspect LINCS/CMap metadata table schemas
# =============================================================================

metadata_preview_paths = {
    "cellinfo": LINCS_CELLINFO_PATH,
    "compoundinfo": LINCS_COMPOUNDINFO_PATH,
    "geneinfo": LINCS_GENEINFO_PATH,
    "instinfo": LINCS_INSTINFO_PATH,
    "siginfo": LINCS_SIGINFO_PATH,
}

metadata_previews = {
    name: pd.read_csv(path, sep="\t", nrows=5, low_memory=False)
    for name, path in metadata_preview_paths.items()
}

for name, preview in metadata_previews.items():
    print(f"\n{name}")
    print(f"  columns ({len(preview.columns)}):")
    print("  " + ", ".join(preview.columns))

In [ ]:
# =============================================================================
# Inspect LINCS/CMap metadata-definition workbook
# =============================================================================

with pd.ExcelFile(LINCS_METADATA_DEFINITIONS_PATH) as workbook:
    metadata_definition_sheets = workbook.sheet_names

print("Metadata-definition sheets:")
for sheet_name in metadata_definition_sheets:
    print(f"  - {sheet_name}")

In [ ]:
# =============================================================================
# Load LINCS/CMap metadata field definitions
# =============================================================================

metadata_definitions = pd.read_excel(
    LINCS_METADATA_DEFINITIONS_PATH,
    sheet_name=None,
)

for sheet_name, table in metadata_definitions.items():
    print(
        f"{sheet_name}: "
        f"{table.shape[0]:,} rows x {table.shape[1]} columns"
    )
    print(f"  columns: {table.columns.tolist()}")

In [ ]:
# =============================================================================
# Audit metadata schemas against provider field definitions
# =============================================================================

schema_audit_records = []

for table_name in [
    "siginfo",
    "instinfo",
    "cellinfo",
    "geneinfo",
    "compoundinfo",
]:
    observed_fields = set(metadata_previews[table_name].columns)

    documented_fields = set(
        metadata_definitions[table_name]["Field Name"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    schema_audit_records.append(
        {
            "table": table_name,
            "observed_fields": len(observed_fields),
            "documented_fields": len(documented_fields),
            "observed_not_documented": sorted(
                observed_fields - documented_fields
            ),
            "documented_not_observed": sorted(
                documented_fields - observed_fields
            ),
        }
    )

metadata_schema_audit = pd.DataFrame(schema_audit_records)

metadata_schema_audit

In [ ]:
# =============================================================================
# Report metadata schema discrepancies
# =============================================================================

for row in metadata_schema_audit.itertuples(index=False):
    if row.observed_not_documented or row.documented_not_observed:
        print(row.table)

        print("  Observed but not documented:")
        for field in row.observed_not_documented:
            print(f"    - {field}")

        print("  Documented but not observed:")
        for field in row.documented_not_observed:
            print(f"    - {field}")

        print()

In [ ]:
# =============================================================================
# Inspect analysis-relevant siginfo field definitions
# =============================================================================

siginfo_fields_of_interest = [
    "sig_id",
    "pert_id",
    "pert_type",
    "cell_iname",
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
    "nsample",
    "cc_q75",
    "ss_ngene",
    "tas",
    "is_hiq",
    "qc_pass",
    "is_ncs_exemplar",
]

siginfo_definition_subset = (
    metadata_definitions["siginfo"]
    .loc[
        metadata_definitions["siginfo"]["Field Name"].isin(
            siginfo_fields_of_interest
        )
    ]
    .reset_index(drop=True)
)

siginfo_definition_subset

In [ ]:
# =============================================================================
# Inspect gene-space field definitions
# =============================================================================

geneinfo_definition_subset = (
    metadata_definitions["geneinfo"]
    .loc[
        metadata_definitions["geneinfo"]["Field Name"].isin(
            [
                "gene_id",
                "gene_symbol",
                "gene_type",
                "src",
                "feature_space",
            ]
        )
    ]
    .reset_index(drop=True)
)

geneinfo_definition_subset

In [ ]:
# =============================================================================
# Characterize LINCS/CMap gene feature spaces
# =============================================================================

lincs_geneinfo = pd.read_csv(
    LINCS_GENEINFO_PATH,
    sep="\t",
    low_memory=False,
)

gene_feature_space_counts = (
    lincs_geneinfo["feature_space"]
    .value_counts(dropna=False)
    .rename_axis("feature_space")
    .reset_index(name="n_genes")
)

print(
    f"geneinfo: "
    f"{lincs_geneinfo.shape[0]:,} rows x "
    f"{lincs_geneinfo.shape[1]} columns"
)

gene_feature_space_counts

In [ ]:
# =============================================================================
# Validate gene identifiers against Level 5 GCTX rows
# =============================================================================

assert lincs_geneinfo["gene_id"].notna().all()
assert lincs_geneinfo["gene_id"].is_unique

with h5py.File(LINCS_LEVEL5_TRT_CP_PATH, mode="r") as gctx:
    gctx_gene_ids = [
        value.decode("utf-8") if isinstance(value, bytes) else str(value)
        for value in gctx["0/META/ROW/id"][:]
    ]

geneinfo_gene_ids = lincs_geneinfo["gene_id"].astype(str).tolist()

missing_from_gctx = sorted(
    set(geneinfo_gene_ids) - set(gctx_gene_ids)
)

missing_from_geneinfo = sorted(
    set(gctx_gene_ids) - set(geneinfo_gene_ids)
)

print(f"geneinfo unique gene IDs: {len(geneinfo_gene_ids):,}")
print(f"GCTX row gene IDs:        {len(gctx_gene_ids):,}")
print(f"Missing from GCTX:         {len(missing_from_gctx):,}")
print(f"Missing from geneinfo:     {len(missing_from_geneinfo):,}")

In [ ]:
# =============================================================================
# Load frozen Phase 4 consensus transcriptomic gene weights
# =============================================================================

CONSENSUS_GENE_WEIGHTS_PATH = (
    Paths.consensus_programs
    / "401_consensus_transcriptomic_gene_weights.csv"
)

consensus_gene_weights = pd.read_csv(
    CONSENSUS_GENE_WEIGHTS_PATH
)

print(
    f"Consensus gene weights: "
    f"{consensus_gene_weights.shape[0]:,} rows x "
    f"{consensus_gene_weights.shape[1]} columns"
)

print(
    "Programs:",
    sorted(consensus_gene_weights["consensus_program_id"].unique()),
)

In [ ]:
# =============================================================================
# Audit exact gene-symbol compatibility with LINCS/CMap
# =============================================================================

phase4_gene_symbols = (
    consensus_gene_weights["gene_symbol"]
    .dropna()
    .astype(str)
    .unique()
)

lincs_symbol_counts = (
    lincs_geneinfo["gene_symbol"]
    .dropna()
    .astype(str)
    .value_counts()
)

duplicate_lincs_symbols = lincs_symbol_counts[
    lincs_symbol_counts > 1
]

phase4_symbol_set = set(phase4_gene_symbols)
lincs_symbol_set = set(lincs_symbol_counts.index)

exact_overlap = phase4_symbol_set & lincs_symbol_set
missing_from_lincs = phase4_symbol_set - lincs_symbol_set
ambiguous_overlap = exact_overlap & set(duplicate_lincs_symbols.index)

print(f"Unique Phase 4 genes:       {len(phase4_symbol_set):,}")
print(f"Exact LINCS symbol overlap: {len(exact_overlap):,}")
print(f"Missing from LINCS:         {len(missing_from_lincs):,}")
print(f"Ambiguous LINCS symbols:    {len(ambiguous_overlap):,}")

In [ ]:
# =============================================================================
# Annotate Phase 4 consensus weights with LINCS/CMap gene space
# =============================================================================

lincs_gene_space_map = (
    lincs_geneinfo
    .loc[
        lincs_geneinfo["gene_symbol"].isin(phase4_symbol_set),
        [
            "gene_symbol",
            "gene_id",
            "feature_space",
        ],
    ]
    .copy()
)

assert lincs_gene_space_map["gene_symbol"].is_unique

consensus_weights_lincs = consensus_gene_weights.merge(
    lincs_gene_space_map,
    on="gene_symbol",
    how="left",
    validate="many_to_one",
)

consensus_weights_lincs["in_lincs"] = (
    consensus_weights_lincs["feature_space"].notna()
)

consensus_weights_lincs["in_landmark"] = (
    consensus_weights_lincs["feature_space"].eq("landmark")
)

consensus_weights_lincs["in_bing"] = (
    consensus_weights_lincs["feature_space"].isin(
        ["landmark", "best inferred"]
    )
)

gene_space_coverage_counts = (
    consensus_weights_lincs
    .groupby("consensus_program_id", as_index=False)
    .agg(
        total_genes=("gene_symbol", "size"),
        lincs_genes=("in_lincs", "sum"),
        landmark_genes=("in_landmark", "sum"),
        bing_genes=("in_bing", "sum"),
    )
)

gene_space_coverage_counts

In [ ]:
# =============================================================================
# Quantify consensus-weight retention across LINCS/CMap gene spaces
# =============================================================================

weight_retention_records = []

for program_id, group in consensus_weights_lincs.groupby(
    "consensus_program_id",
    sort=True,
):
    weights = group["consensus_weight"].to_numpy(dtype=float)

    total_l1 = np.abs(weights).sum()
    total_l2_sq = np.square(weights).sum()

    record = {
        "consensus_program_id": program_id,
    }

    for space_name, mask_column in [
        ("landmark", "in_landmark"),
        ("bing", "in_bing"),
        ("lincs_all", "in_lincs"),
    ]:
        mask = group[mask_column].to_numpy(dtype=bool)
        retained_weights = weights[mask]

        record[f"{space_name}_l1_fraction"] = (
            np.abs(retained_weights).sum() / total_l1
        )

        record[f"{space_name}_l2_sq_fraction"] = (
            np.square(retained_weights).sum() / total_l2_sq
        )

    weight_retention_records.append(record)

gene_space_weight_retention = pd.DataFrame(
    weight_retention_records
)

gene_space_weight_retention

In [ ]:
# =============================================================================
# Audit signed-arm weight retention across LINCS/CMap gene spaces
# =============================================================================

signed_retention_records = []

for program_id, group in consensus_weights_lincs.groupby(
    "consensus_program_id",
    sort=True,
):
    weights = group["consensus_weight"].to_numpy(dtype=float)

    positive_mask = weights > 0
    negative_mask = weights < 0

    total_positive_l1 = np.abs(weights[positive_mask]).sum()
    total_negative_l1 = np.abs(weights[negative_mask]).sum()

    record = {
        "consensus_program_id": program_id,
    }

    for space_name, mask_column in [
        ("landmark", "in_landmark"),
        ("bing", "in_bing"),
        ("lincs_all", "in_lincs"),
    ]:
        space_mask = group[mask_column].to_numpy(dtype=bool)

        record[f"{space_name}_positive_l1_fraction"] = (
            np.abs(weights[positive_mask & space_mask]).sum()
            / total_positive_l1
        )

        record[f"{space_name}_negative_l1_fraction"] = (
            np.abs(weights[negative_mask & space_mask]).sum()
            / total_negative_l1
        )

    signed_retention_records.append(record)

signed_arm_weight_retention = pd.DataFrame(
    signed_retention_records
)

signed_arm_weight_retention

In [ ]:
# =============================================================================
# Summarize LINCS/CMap representation fidelity
# =============================================================================

representation_fidelity = (
    gene_space_weight_retention
    .merge(
        signed_arm_weight_retention,
        on="consensus_program_id",
        validate="one_to_one",
    )
    .copy()
)

for space_name in [
    "landmark",
    "bing",
    "lincs_all",
]:
    representation_fidelity[
        f"{space_name}_cosine_fidelity"
    ] = np.sqrt(
        representation_fidelity[
            f"{space_name}_l2_sq_fraction"
        ]
    )

    representation_fidelity[
        f"{space_name}_minimum_signed_arm_l1_fraction"
    ] = representation_fidelity[
        [
            f"{space_name}_positive_l1_fraction",
            f"{space_name}_negative_l1_fraction",
        ]
    ].min(axis=1)

representation_fidelity[
    [
        "consensus_program_id",
        "landmark_cosine_fidelity",
        "landmark_minimum_signed_arm_l1_fraction",
        "bing_cosine_fidelity",
        "bing_minimum_signed_arm_l1_fraction",
        "lincs_all_cosine_fidelity",
        "lincs_all_minimum_signed_arm_l1_fraction",
    ]
]

In [ ]:
# =============================================================================
# Load signature identifiers for Level 5 metadata audit
# =============================================================================

lincs_siginfo_ids = pd.read_csv(
    LINCS_SIGINFO_PATH,
    sep="\t",
    usecols=[
        "sig_id",
        "pert_type",
    ],
    low_memory=False,
)

with h5py.File(LINCS_LEVEL5_TRT_CP_PATH, mode="r") as gctx:
    gctx_signature_ids = [
        value.decode("utf-8") if isinstance(value, bytes) else str(value)
        for value in gctx["0/META/COL/id"][:]
    ]

print(f"siginfo rows:             {len(lincs_siginfo_ids):,}")
print(f"siginfo unique sig_id:    {lincs_siginfo_ids['sig_id'].nunique():,}")
print(f"GCTX signature IDs:       {len(gctx_signature_ids):,}")
print(f"GCTX unique signature IDs:{len(set(gctx_signature_ids)):,}")

In [ ]:
# =============================================================================
# Validate Level 5 GCTX signature metadata correspondence
# =============================================================================

gctx_signature_id_set = set(gctx_signature_ids)

gctx_siginfo = (
    lincs_siginfo_ids
    .loc[
        lincs_siginfo_ids["sig_id"].isin(
            gctx_signature_id_set
        )
    ]
    .copy()
)

matched_sig_id_set = set(gctx_siginfo["sig_id"])

missing_from_siginfo = (
    gctx_signature_id_set - matched_sig_id_set
)

siginfo_trt_cp_ids = set(
    lincs_siginfo_ids.loc[
        lincs_siginfo_ids["pert_type"].eq("trt_cp"),
        "sig_id",
    ]
)

trt_cp_missing_from_gctx = (
    siginfo_trt_cp_ids - gctx_signature_id_set
)

print(f"GCTX signatures matched in siginfo: {len(matched_sig_id_set):,}")
print(f"GCTX signatures missing from siginfo: {len(missing_from_siginfo):,}")
print()
print("pert_type among GCTX signatures:")
print(gctx_siginfo["pert_type"].value_counts(dropna=False))
print()
print(f"siginfo trt_cp signatures:          {len(siginfo_trt_cp_ids):,}")
print(f"trt_cp signatures absent from GCTX: {len(trt_cp_missing_from_gctx):,}")

In [ ]:
# =============================================================================
# Load Level 5 trt_cp signature metadata for coverage audit
# =============================================================================

siginfo_audit_columns = [
    "sig_id",
    "pert_id",
    "cmap_name",
    "cell_iname",
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
    "nsample",
    "cc_q75",
    "ss_ngene",
    "tas",
    "qc_pass",
    "is_hiq",
    "project_code",
    "build_name",
]

lincs_trt_cp_siginfo = (
    pd.read_csv(
        LINCS_SIGINFO_PATH,
        sep="\t",
        usecols=siginfo_audit_columns,
        low_memory=False,
    )
    .loc[
        lambda df: df["sig_id"].isin(gctx_signature_id_set)
    ]
    .reset_index(drop=True)
)

print(
    f"Level 5 trt_cp metadata: "
    f"{lincs_trt_cp_siginfo.shape[0]:,} rows x "
    f"{lincs_trt_cp_siginfo.shape[1]} columns"
)

print(
    f"Unique signatures: "
    f"{lincs_trt_cp_siginfo['sig_id'].nunique():,}"
)
print(
    f"Unique perturbagens: "
    f"{lincs_trt_cp_siginfo['pert_id'].nunique():,}"
)
print(
    f"Unique cell lines: "
    f"{lincs_trt_cp_siginfo['cell_iname'].nunique():,}"
)

In [ ]:
# =============================================================================
# Audit cell-line metadata coverage for Level 5 trt_cp signatures
# =============================================================================

lincs_cellinfo = pd.read_csv(
    LINCS_CELLINFO_PATH,
    sep="\t",
    low_memory=False,
)

assert lincs_cellinfo["cell_iname"].is_unique

trt_cp_cell_lines = set(
    lincs_trt_cp_siginfo["cell_iname"].dropna()
)

cellinfo_cell_lines = set(
    lincs_cellinfo["cell_iname"].dropna()
)

missing_cell_metadata = (
    trt_cp_cell_lines - cellinfo_cell_lines
)

trt_cp_cellinfo = (
    lincs_cellinfo
    .loc[
        lincs_cellinfo["cell_iname"].isin(
            trt_cp_cell_lines
        )
    ]
    .copy()
)

print(f"trt_cp cell lines:             {len(trt_cp_cell_lines):,}")
print(f"Matched in cellinfo:           {len(trt_cp_cellinfo):,}")
print(f"Missing from cellinfo:         {len(missing_cell_metadata):,}")
print(
    "Missing cell_lineage values: "
    f"{trt_cp_cellinfo['cell_lineage'].isna().sum():,}"
)
print(
    "Unique non-null lineages:    "
    f"{trt_cp_cellinfo['cell_lineage'].nunique(dropna=True):,}"
)

In [ ]:
# =============================================================================
# Characterize Level 5 trt_cp cell-line composition
# =============================================================================

lineage_cell_counts = (
    trt_cp_cellinfo
    .groupby("cell_lineage", dropna=False)["cell_iname"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_cell_lines")
    .reset_index()
)

cell_type_counts = (
    trt_cp_cellinfo
    .groupby("cell_type", dropna=False)["cell_iname"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_cell_lines")
    .reset_index()
)

print("Cell lines by lineage:")
display(lineage_cell_counts)

print("\nCell lines by cell type:")
display(cell_type_counts)

In [ ]:
# =============================================================================
# Cross-tabulate cell type and lineage annotation
# =============================================================================

cell_type_lineage_counts = (
    trt_cp_cellinfo
    .groupby(
        ["cell_type", "cell_lineage"],
        dropna=False,
    )["cell_iname"]
    .nunique()
    .rename("n_cell_lines")
    .reset_index()
    .sort_values(
        ["cell_type", "n_cell_lines", "cell_lineage"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

display(cell_type_lineage_counts)

print("\nUnknown-lineage cell lines by cell type:")
display(
    cell_type_lineage_counts.loc[
        cell_type_lineage_counts["cell_lineage"].eq("unknown")
    ]
)

In [ ]:
# =============================================================================
# Inspect metadata for tumor cell lines with unknown lineage
# =============================================================================

unknown_tumor_cellinfo = (
    trt_cp_cellinfo
    .loc[
        trt_cp_cellinfo["cell_type"].eq("tumor")
        & trt_cp_cellinfo["cell_lineage"].eq("unknown"),
        [
            "cell_iname",
            "primary_disease",
            "subtype",
            "ccle_name",
            "cellosaurus_id",
        ],
    ]
    .copy()
)

print(
    f"Tumor cell lines with unknown lineage: "
    f"{len(unknown_tumor_cellinfo):,}"
)

print("\nPrimary disease:")
display(
    unknown_tumor_cellinfo["primary_disease"]
    .value_counts(dropna=False)
    .rename_axis("primary_disease")
    .reset_index(name="n_cell_lines")
)

print("\nSubtype missingness:")
print(
    unknown_tumor_cellinfo["subtype"]
    .isna()
    .value_counts()
)

In [ ]:
# =============================================================================
# Summarize informative annotations for tumor cell lines with unknown lineage
# =============================================================================

unknown_tumor_subtype_counts = (
    unknown_tumor_cellinfo["subtype"]
    .value_counts(dropna=False)
    .rename_axis("subtype")
    .reset_index(name="n_cell_lines")
)

informative_unknown_tumor_annotations = (
    unknown_tumor_cellinfo
    .loc[
        (
            unknown_tumor_cellinfo["subtype"].notna()
            & ~unknown_tumor_cellinfo["subtype"].eq("unknown")
        )
        | unknown_tumor_cellinfo["ccle_name"].notna()
        | unknown_tumor_cellinfo["cellosaurus_id"].notna()
    ]
    .sort_values(["subtype", "cell_iname"])
    .reset_index(drop=True)
)

display(unknown_tumor_subtype_counts)

print(
    "\nUnknown-lineage tumor cell lines with at least one "
    "informative subtype/external-ID field: "
    f"{len(informative_unknown_tumor_annotations):,}"
)

display(informative_unknown_tumor_annotations)


In [ ]:
# =============================================================================
# Quantify signature coverage by cell-context category
# =============================================================================

cell_context_map = (
    trt_cp_cellinfo[
        [
            "cell_iname",
            "cell_type",
            "cell_lineage",
        ]
    ]
    .copy()
)

lincs_trt_cp_context = lincs_trt_cp_siginfo.merge(
    cell_context_map,
    on="cell_iname",
    how="left",
    validate="many_to_one",
)

lincs_trt_cp_context["context_category"] = np.select(
    [
        lincs_trt_cp_context["cell_type"].eq("tumor")
        & ~lincs_trt_cp_context["cell_lineage"].eq("unknown"),

        lincs_trt_cp_context["cell_type"].eq("tumor")
        & lincs_trt_cp_context["cell_lineage"].eq("unknown"),

        lincs_trt_cp_context["cell_type"].eq("normal"),

        lincs_trt_cp_context["cell_type"].eq("pool"),
    ],
    [
        "tumor_known_lineage",
        "tumor_unknown_lineage",
        "normal",
        "pool",
    ],
    default="other",
)

context_coverage_summary = (
    lincs_trt_cp_context
    .groupby("context_category", as_index=False)
    .agg(
        n_signatures=("sig_id", "nunique"),
        n_perturbagens=("pert_id", "nunique"),
        n_cell_lines=("cell_iname", "nunique"),
    )
    .sort_values("n_signatures", ascending=False)
    .reset_index(drop=True)
)

context_coverage_summary

In [ ]:
# =============================================================================
# Characterize perturbagen coverage across tumor lineages
# =============================================================================

tumor_known_lineage_signatures = (
    lincs_trt_cp_context
    .loc[
        lincs_trt_cp_context["context_category"]
        .eq("tumor_known_lineage")
    ]
    .copy()
)

perturbagen_context_coverage = (
    tumor_known_lineage_signatures
    .groupby("pert_id", as_index=False)
    .agg(
        n_signatures=("sig_id", "nunique"),
        n_cell_lines=("cell_iname", "nunique"),
        n_lineages=("cell_lineage", "nunique"),
    )
)

coverage_distribution = (
    perturbagen_context_coverage[
        [
            "n_signatures",
            "n_cell_lines",
            "n_lineages",
        ]
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

coverage_distribution

In [ ]:
# =============================================================================
# Evaluate candidate two-dimensional coverage gates
# =============================================================================

cell_line_thresholds = [2, 3, 4, 5, 6, 8, 10]
lineage_thresholds = [2, 3, 4, 5, 6]

coverage_gate_grid = pd.DataFrame(
    {
        min_lineages: [
            (
                (perturbagen_context_coverage["n_cell_lines"] >= min_cell_lines)
                & (
                    perturbagen_context_coverage["n_lineages"]
                    >= min_lineages
                )
            ).sum()
            for min_cell_lines in cell_line_thresholds
        ]
        for min_lineages in lineage_thresholds
    },
    index=cell_line_thresholds,
)

coverage_gate_grid.index.name = "min_cell_lines"
coverage_gate_grid.columns.name = "min_lineages"

coverage_gate_grid

In [ ]:
# =============================================================================
# Express candidate coverage gates as retained perturbagen fractions
# =============================================================================

n_tumor_known_perturbagens = (
    perturbagen_context_coverage["pert_id"].nunique()
)

coverage_gate_fraction_grid = (
    coverage_gate_grid
    / n_tumor_known_perturbagens
    * 100
)

coverage_gate_fraction_grid.round(1)

In [ ]:
# =============================================================================
# Compare experimental support under candidate coverage gates
# =============================================================================

candidate_gates = [
    (4, 4),
    (5, 4),
    (5, 5),
]

gate_support_records = []

for min_cell_lines, min_lineages in candidate_gates:
    eligible = perturbagen_context_coverage.loc[
        (
            perturbagen_context_coverage["n_cell_lines"]
            >= min_cell_lines
        )
        & (
            perturbagen_context_coverage["n_lineages"]
            >= min_lineages
        )
    ]

    gate_support_records.append(
        {
            "min_cell_lines": min_cell_lines,
            "min_lineages": min_lineages,
            "n_perturbagens": len(eligible),
            "retained_pct": (
                len(eligible)
                / n_tumor_known_perturbagens
                * 100
            ),
            "median_signatures": eligible["n_signatures"].median(),
            "median_cell_lines": eligible["n_cell_lines"].median(),
            "median_lineages": eligible["n_lineages"].median(),
            "p10_cell_lines": eligible["n_cell_lines"].quantile(0.10),
            "p10_lineages": eligible["n_lineages"].quantile(0.10),
        }
    )

candidate_gate_support = pd.DataFrame(
    gate_support_records
)

candidate_gate_support

In [ ]:
# =============================================================================
# Characterize provider QC flags in tumor known-lineage signatures
# =============================================================================

qc_flag_summary = (
    tumor_known_lineage_signatures
    .groupby(
        ["qc_pass", "is_hiq"],
        dropna=False,
    )
    .agg(
        n_signatures=("sig_id", "nunique"),
        n_perturbagens=("pert_id", "nunique"),
        n_cell_lines=("cell_iname", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["qc_pass", "is_hiq"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

display(qc_flag_summary)

print("\nMissingness:")
print(
    tumor_known_lineage_signatures[
        ["qc_pass", "is_hiq"]
    ]
    .isna()
    .sum()
)

In [ ]:
# =============================================================================
# Inspect provider definitions for signature QC flags
# =============================================================================

for field_name in ["qc_pass", "is_hiq"]:
    row = metadata_definitions["siginfo"].loc[
        metadata_definitions["siginfo"]["Field Name"].eq(field_name)
    ].iloc[0]

    print(field_name)
    print(f"  Type:        {row['Type']}")
    print(f"  Example:     {row['Example']}")
    print(f"  Description: {row['Description']}")
    print()

In [ ]:
# =============================================================================
# Inspect provider definition of instance-level qc_pass
# =============================================================================

instinfo_qc_definition = (
    metadata_definitions["instinfo"]
    .loc[
        metadata_definitions["instinfo"]["Field Name"]
        .eq("qc_pass")
    ]
    .iloc[0]
)

print("qc_pass")
print(f"  Type:        {instinfo_qc_definition['Type']}")
print(f"  Example:     {instinfo_qc_definition['Example']}")
print(f"  Description: {instinfo_qc_definition['Description']}")

In [ ]:
# =============================================================================
# Recalculate perturbagen coverage after provider QC
# =============================================================================

tumor_known_lineage_qc = (
    tumor_known_lineage_signatures
    .loc[
        tumor_known_lineage_signatures["qc_pass"].eq(1)
    ]
    .copy()
)

perturbagen_context_coverage_qc = (
    tumor_known_lineage_qc
    .groupby("pert_id", as_index=False)
    .agg(
        n_signatures=("sig_id", "nunique"),
        n_cell_lines=("cell_iname", "nunique"),
        n_lineages=("cell_lineage", "nunique"),
    )
)

print(
    f"QC-passing signatures: "
    f"{len(tumor_known_lineage_qc):,}"
)
print(
    f"QC-passing perturbagens: "
    f"{len(perturbagen_context_coverage_qc):,}"
)

perturbagen_context_coverage_qc[
    [
        "n_signatures",
        "n_cell_lines",
        "n_lineages",
    ]
].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

In [ ]:
# =============================================================================
# Evaluate coverage gates after provider QC
# =============================================================================

coverage_gate_grid_qc = pd.DataFrame(
    {
        min_lineages: [
            (
                (
                    perturbagen_context_coverage_qc["n_cell_lines"]
                    >= min_cell_lines
                )
                & (
                    perturbagen_context_coverage_qc["n_lineages"]
                    >= min_lineages
                )
            ).sum()
            for min_cell_lines in cell_line_thresholds
        ]
        for min_lineages in lineage_thresholds
    },
    index=cell_line_thresholds,
)

coverage_gate_grid_qc.index.name = "min_cell_lines"
coverage_gate_grid_qc.columns.name = "min_lineages"

coverage_gate_fraction_grid_qc = (
    coverage_gate_grid_qc
    / len(perturbagen_context_coverage_qc)
    * 100
)

print("Eligible perturbagens:")
display(coverage_gate_grid_qc)

print("\nRetained perturbagens (%):")
display(coverage_gate_fraction_grid_qc.round(1))

In [ ]:
# =============================================================================
# Compare candidate coverage gates after provider QC
# =============================================================================

candidate_gates_qc = [
    (4, 4),
    (5, 4),
    (5, 5),
]

gate_support_records_qc = []

for min_cell_lines, min_lineages in candidate_gates_qc:
    eligible = perturbagen_context_coverage_qc.loc[
        (
            perturbagen_context_coverage_qc["n_cell_lines"]
            >= min_cell_lines
        )
        & (
            perturbagen_context_coverage_qc["n_lineages"]
            >= min_lineages
        )
    ]

    gate_support_records_qc.append(
        {
            "min_cell_lines": min_cell_lines,
            "min_lineages": min_lineages,
            "n_perturbagens": len(eligible),
            "retained_pct": (
                len(eligible)
                / len(perturbagen_context_coverage_qc)
                * 100
            ),
            "median_signatures": eligible["n_signatures"].median(),
            "median_cell_lines": eligible["n_cell_lines"].median(),
            "median_lineages": eligible["n_lineages"].median(),
            "p10_signatures": eligible["n_signatures"].quantile(0.10),
            "p10_cell_lines": eligible["n_cell_lines"].quantile(0.10),
            "p10_lineages": eligible["n_lineages"].quantile(0.10),
        }
    )

candidate_gate_support_qc = pd.DataFrame(
    gate_support_records_qc
)

candidate_gate_support_qc

## Frozen primary technical coverage gate

The primary Phase 7 perturbational universe will be defined using Level 5
`trt_cp` signatures that satisfy the provider's standard technical QC
(`qc_pass == 1`) in tumor cell lines with an explicitly annotated lineage.

A perturbagen will be considered evaluable for the primary lineage-aware
analysis only if its QC-passing signatures cover:

- at least **5 unique tumor cell lines**, and
- at least **4 unique annotated lineages**.

This gate was selected outcome-blind, before calculating any connectivity
scores. In the audited LINCS 2020 Beta Level 5 `trt_cp` resource, it retains
7,915 of 29,799 perturbagens represented by at least one QC-passing
tumor/known-lineage signature (26.6%).

The two-dimensional requirement avoids treating compounds represented in only
a few biological contexts as cross-lineage perturbational evidence. The nearby
4-cell-line/4-lineage gate retains 8,040 perturbagens, whereas the stricter
5-cell-line/5-lineage gate retains 7,641; their lower-tail experimental support
is otherwise very similar.

`is_hiq` is not used as a primary inclusion criterion because it additionally
selects signatures using functional recall metrics. It will instead be retained
for a stricter quality-sensitivity analysis.

Tumor signatures lacking an annotated lineage are not manually reassigned and
are excluded from the primary lineage-balanced aggregation. Normal and pooled
cell contexts are also outside the primary universe and may be examined
separately as contextual sensitivity analyses.

In [ ]:
# =============================================================================
# Freeze primary Phase 7 technical coverage criteria
# =============================================================================

PRIMARY_CELL_TYPE = "tumor"
PRIMARY_EXCLUDED_LINEAGE = "unknown"
PRIMARY_QC_PASS = 1
PRIMARY_MIN_CELL_LINES = 5
PRIMARY_MIN_LINEAGES = 4

primary_evaluable_perturbagens = (
    perturbagen_context_coverage_qc
    .loc[
        (
            perturbagen_context_coverage_qc["n_cell_lines"]
            >= PRIMARY_MIN_CELL_LINES
        )
        & (
            perturbagen_context_coverage_qc["n_lineages"]
            >= PRIMARY_MIN_LINEAGES
        )
    ]
    .copy()
    .sort_values("pert_id")
    .reset_index(drop=True)
)

primary_evaluable_perturbagen_ids = set(
    primary_evaluable_perturbagens["pert_id"]
)

print(
    f"Primary evaluable perturbagens: "
    f"{len(primary_evaluable_perturbagens):,}"
)

primary_evaluable_perturbagens[
    [
        "n_signatures",
        "n_cell_lines",
        "n_lineages",
    ]
].describe()

In [ ]:
# =============================================================================
# Audit dose, time, and replicate metadata in the primary evaluable universe
# =============================================================================

primary_evaluable_signatures = (
    tumor_known_lineage_qc
    .loc[
        tumor_known_lineage_qc["pert_id"].isin(
            primary_evaluable_perturbagen_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Primary evaluable signatures: "
    f"{len(primary_evaluable_signatures):,}"
)

print("\nMetadata missingness:")
print(
    primary_evaluable_signatures[
        [
            "pert_dose",
            "pert_dose_unit",
            "pert_time",
            "pert_time_unit",
            "nsample",
        ]
    ]
    .isna()
    .sum()
)

print("\nDose units:")
print(
    primary_evaluable_signatures["pert_dose_unit"]
    .value_counts(dropna=False)
)

print("\nTime units:")
print(
    primary_evaluable_signatures["pert_time_unit"]
    .value_counts(dropna=False)
)

print("\nReplicates per Level 5 signature:")
print(
    primary_evaluable_signatures["nsample"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
# =============================================================================
# Inspect signatures with incomplete condition metadata
# =============================================================================

condition_metadata_columns = [
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
]

incomplete_condition_signatures = (
    primary_evaluable_signatures
    .loc[
        primary_evaluable_signatures[
            condition_metadata_columns
        ].isna().any(axis=1)
    ]
    .copy()
)

print(
    f"Signatures with incomplete condition metadata: "
    f"{len(incomplete_condition_signatures):,}"
)
print(
    f"Affected perturbagens: "
    f"{incomplete_condition_signatures['pert_id'].nunique():,}"
)
print(
    f"Affected cell lines: "
    f"{incomplete_condition_signatures['cell_iname'].nunique():,}"
)

print("\nMissing values by condition field:")
print(
    incomplete_condition_signatures[
        condition_metadata_columns
    ]
    .isna()
    .sum()
)

print("\nFirst affected signatures:")
display(
    incomplete_condition_signatures[
        [
            "pert_id",
            "cmap_name",
            "cell_iname",
            "pert_dose",
            "pert_dose_unit",
            "pert_time",
            "pert_time_unit",
            "nsample",
        ]
    ]
    .sort_values(
        ["pert_id", "cell_iname", "pert_time"]
    )
    .head(20)
    .reset_index(drop=True)
)


In [ ]:
# =============================================================================
# Assess impact of requiring complete condition metadata
# =============================================================================

complete_condition_mask = (
    primary_evaluable_signatures[
        condition_metadata_columns
    ]
    .notna()
    .all(axis=1)
)

primary_complete_condition_signatures = (
    primary_evaluable_signatures
    .loc[complete_condition_mask]
    .copy()
)

coverage_after_complete_conditions = (
    primary_complete_condition_signatures
    .groupby("pert_id", as_index=False)
    .agg(
        n_signatures=("sig_id", "nunique"),
        n_cell_lines=("cell_iname", "nunique"),
        n_lineages=("cell_lineage", "nunique"),
    )
)

coverage_after_complete_conditions["still_evaluable"] = (
    (
        coverage_after_complete_conditions["n_cell_lines"]
        >= PRIMARY_MIN_CELL_LINES
    )
    & (
        coverage_after_complete_conditions["n_lineages"]
        >= PRIMARY_MIN_LINEAGES
    )
)

lost_evaluable_perturbagens = (
    primary_evaluable_perturbagen_ids
    - set(
        coverage_after_complete_conditions.loc[
            coverage_after_complete_conditions["still_evaluable"],
            "pert_id",
        ]
    )
)

print(
    f"Signatures retained: "
    f"{len(primary_complete_condition_signatures):,} / "
    f"{len(primary_evaluable_signatures):,}"
)
print(
    f"Evaluable perturbagens retained: "
    f"{coverage_after_complete_conditions['still_evaluable'].sum():,} / "
    f"{len(primary_evaluable_perturbagen_ids):,}"
)
print(
    f"Perturbagens losing evaluability: "
    f"{len(lost_evaluable_perturbagens):,}"
)

if lost_evaluable_perturbagens:
    display(
        coverage_after_complete_conditions.loc[
            coverage_after_complete_conditions["pert_id"].isin(
                lost_evaluable_perturbagens
            )
        ]
    )


In [ ]:
# =============================================================================
# Inspect perturbagens losing evaluability after condition completeness filter
# =============================================================================

lost_evaluable_comparison = (
    primary_evaluable_perturbagens
    .loc[
        primary_evaluable_perturbagens["pert_id"].isin(
            lost_evaluable_perturbagens
        )
    ]
    .rename(
        columns={
            "n_signatures": "n_signatures_before",
            "n_cell_lines": "n_cell_lines_before",
            "n_lineages": "n_lineages_before",
        }
    )
    .merge(
        coverage_after_complete_conditions[
            [
                "pert_id",
                "n_signatures",
                "n_cell_lines",
                "n_lineages",
            ]
        ].rename(
            columns={
                "n_signatures": "n_signatures_after",
                "n_cell_lines": "n_cell_lines_after",
                "n_lineages": "n_lineages_after",
            }
        ),
        on="pert_id",
        how="left",
        validate="one_to_one",
    )
)

perturbagen_names = (
    primary_evaluable_signatures[
        ["pert_id", "cmap_name"]
    ]
    .drop_duplicates()
)

lost_evaluable_comparison = (
    lost_evaluable_comparison
    .merge(
        perturbagen_names,
        on="pert_id",
        how="left",
        validate="one_to_one",
    )
)

lost_evaluable_comparison


In [ ]:
# =============================================================================
# Inspect compound metadata for perturbagens losing final evaluability
# =============================================================================

lincs_compoundinfo = pd.read_csv(
    LINCS_COMPOUNDINFO_PATH,
    sep="\t",
    low_memory=False,
)

lost_evaluable_compoundinfo = (
    lincs_compoundinfo
    .loc[
        lincs_compoundinfo["pert_id"].isin(
            lost_evaluable_perturbagens
        )
    ]
    .sort_values(["pert_id", "cmap_name"])
    .reset_index(drop=True)
)

lost_evaluable_compoundinfo


In [ ]:
# =============================================================================
# Inspect provider perturbation-type definitions
# =============================================================================

pert_type_definitions = (
    metadata_definitions["pert_type"]
    .copy()
    .reset_index(drop=True)
)

pert_type_definitions

In [ ]:
# =============================================================================
# Audit DMSO perturbation-type annotation in instance metadata
# =============================================================================

DMSO_PERT_ID = "BRD-K08970894"

dmso_instinfo_chunks = []

for chunk in pd.read_csv(
    LINCS_INSTINFO_PATH,
    sep="\t",
    usecols=[
        "pert_id",
        "pert_type",
        "cmap_name",
        "cell_iname",
    ],
    chunksize=250_000,
    low_memory=False,
):
    dmso_rows = chunk.loc[
        chunk["pert_id"].eq(DMSO_PERT_ID)
    ]

    if not dmso_rows.empty:
        dmso_instinfo_chunks.append(dmso_rows)

dmso_instinfo = pd.concat(
    dmso_instinfo_chunks,
    ignore_index=True,
)

print(f"DMSO instances: {len(dmso_instinfo):,}")

print("\npert_type:")
print(dmso_instinfo["pert_type"].value_counts(dropna=False))

print("\ncmap_name:")
print(dmso_instinfo["cmap_name"].value_counts(dropna=False))

In [ ]:
# =============================================================================
# Freeze complete-condition requirement and final primary universe
# =============================================================================

PRIMARY_REQUIRE_COMPLETE_CONDITION = True

primary_analysis_signatures = (
    primary_complete_condition_signatures
    .copy()
    .reset_index(drop=True)
)

final_primary_coverage = (
    primary_analysis_signatures
    .groupby("pert_id", as_index=False)
    .agg(
        n_signatures=("sig_id", "nunique"),
        n_cell_lines=("cell_iname", "nunique"),
        n_lineages=("cell_lineage", "nunique"),
    )
)

final_primary_coverage = (
    final_primary_coverage
    .loc[
        (
            final_primary_coverage["n_cell_lines"]
            >= PRIMARY_MIN_CELL_LINES
        )
        & (
            final_primary_coverage["n_lineages"]
            >= PRIMARY_MIN_LINEAGES
        )
    ]
    .sort_values("pert_id")
    .reset_index(drop=True)
)

final_primary_perturbagen_ids = set(
    final_primary_coverage["pert_id"]
)

primary_analysis_signatures = (
    primary_analysis_signatures
    .loc[
        primary_analysis_signatures["pert_id"].isin(
            final_primary_perturbagen_ids
        )
    ]
    .reset_index(drop=True)
)

print(
    f"Final primary perturbagens: "
    f"{len(final_primary_perturbagen_ids):,}"
)
print(
    f"Final primary signatures:   "
    f"{len(primary_analysis_signatures):,}"
)


In [ ]:
# =============================================================================
# Quantify stricter is_hiq sensitivity coverage under the frozen primary gate
# =============================================================================

hiq_sensitivity_signatures = (
    primary_analysis_signatures
    .loc[
        primary_analysis_signatures["is_hiq"].eq(1)
    ]
    .copy()
)

hiq_sensitivity_coverage = (
    hiq_sensitivity_signatures
    .groupby("pert_id", as_index=False)
    .agg(
        n_signatures=("sig_id", "nunique"),
        n_cell_lines=("cell_iname", "nunique"),
        n_lineages=("cell_lineage", "nunique"),
    )
)

hiq_sensitivity_coverage["sensitivity_evaluable"] = (
    (
        hiq_sensitivity_coverage["n_cell_lines"]
        >= PRIMARY_MIN_CELL_LINES
    )
    & (
        hiq_sensitivity_coverage["n_lineages"]
        >= PRIMARY_MIN_LINEAGES
    )
)

hiq_sensitivity_evaluable = (
    hiq_sensitivity_coverage
    .loc[
        hiq_sensitivity_coverage["sensitivity_evaluable"]
    ]
    .sort_values("pert_id")
    .reset_index(drop=True)
)

print(
    f"is_hiq signatures within final primary universe: "
    f"{len(hiq_sensitivity_signatures):,}"
)
print(
    f"Perturbagens with >=1 is_hiq signature: "
    f"{hiq_sensitivity_coverage['pert_id'].nunique():,}"
)
print(
    f"Perturbagens retaining the same "
    f"{PRIMARY_MIN_CELL_LINES}-cell / "
    f"{PRIMARY_MIN_LINEAGES}-lineage gate under is_hiq: "
    f"{len(hiq_sensitivity_evaluable):,}"
)

display(
    hiq_sensitivity_evaluable[
        [
            "n_signatures",
            "n_cell_lines",
            "n_lineages",
        ]
    ]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)


## Frozen stricter-quality sensitivity rule

The provider-defined `is_hiq` flag is retained as a stricter technical
sensitivity analysis rather than as a primary eligibility criterion.

For this sensitivity, signatures must still satisfy every primary technical
requirement, including complete condition metadata. A perturbagen is evaluable
only if its `is_hiq == 1` subset independently retains the same minimum
coverage of 5 tumor cell lines and 4 annotated lineages.

This sensitivity cannot add, rescue, or redefine perturbagens that are not
evaluable in the primary analysis. Its coverage is quantified here
outcome-blind, before any connectivity result is inspected.


In [ ]:
# =============================================================================
# Characterize dose and time conditions in the final primary universe
# =============================================================================

print("Perturbation times:")
display(
    primary_analysis_signatures["pert_time"]
    .value_counts()
    .sort_index()
    .rename_axis("pert_time_h")
    .reset_index(name="n_signatures")
)

print("\nDose summary (uM):")
display(
    primary_analysis_signatures["pert_dose"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

print("\nMost frequent doses (uM):")
display(
    primary_analysis_signatures["pert_dose"]
    .value_counts()
    .head(20)
    .rename_axis("pert_dose_uM")
    .reset_index(name="n_signatures")
)

In [ ]:
# =============================================================================
# Audit multiplicity of Level 5 signatures per experimental condition
# =============================================================================

condition_columns = [
    "pert_id",
    "cell_iname",
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
]

condition_multiplicity = (
    primary_analysis_signatures
    .groupby(
        condition_columns,
        dropna=False,
    )
    .size()
    .rename("n_level5_signatures")
    .reset_index()
)

print(
    f"Unique compound × cell × dose × time conditions: "
    f"{len(condition_multiplicity):,}"
)
print(
    f"Conditions with >1 Level 5 signature: "
    f"{(condition_multiplicity['n_level5_signatures'] > 1).sum():,}"
)
print(
    f"Maximum Level 5 signatures per condition: "
    f"{condition_multiplicity['n_level5_signatures'].max():,}"
)

print("\nMultiplicity distribution:")
display(
    condition_multiplicity["n_level5_signatures"]
    .value_counts()
    .sort_index()
    .rename_axis("n_level5_signatures")
    .reset_index(name="n_conditions")
)

In [ ]:
# =============================================================================
# Characterize sources of Level 5 condition multiplicity
# =============================================================================

duplicated_condition_keys = (
    condition_multiplicity
    .loc[
        condition_multiplicity["n_level5_signatures"] > 1,
        condition_columns,
    ]
)

duplicated_condition_signatures = (
    primary_analysis_signatures
    .merge(
        duplicated_condition_keys,
        on=condition_columns,
        how="inner",
        validate="many_to_many",
    )
)

duplicated_condition_structure = (
    duplicated_condition_signatures
    .groupby(
        condition_columns,
        dropna=False,
    )
    .agg(
        n_level5_signatures=("sig_id", "nunique"),
        n_projects=("project_code", "nunique"),
        n_builds=("build_name", "nunique"),
        total_distilled_replicates=("nsample", "sum"),
    )
    .reset_index()
)

n_resolved_by_project = (
    duplicated_condition_structure["n_level5_signatures"]
    == duplicated_condition_structure["n_projects"]
).sum()

n_multi_project = (
    duplicated_condition_structure["n_projects"] > 1
).sum()

n_multi_build = (
    duplicated_condition_structure["n_builds"] > 1
).sum()

print(
    f"Duplicated conditions: "
    f"{len(duplicated_condition_structure):,}"
)
print(
    f"Resolved by project identity: "
    f"{n_resolved_by_project:,}"
)
print(
    f"Conditions spanning >1 project: "
    f"{n_multi_project:,}"
)
print(
    f"Conditions spanning >1 build: "
    f"{n_multi_build:,}"
)

print("\nHighest-multiplicity conditions:")
display(
    duplicated_condition_structure
    .sort_values(
        "n_level5_signatures",
        ascending=False,
    )
    .head(10)
)

In [ ]:
# =============================================================================
# Inspect metadata underlying a high-multiplicity Level 5 condition
# =============================================================================

highest_multiplicity_condition = (
    duplicated_condition_structure
    .sort_values(
        "n_level5_signatures",
        ascending=False,
    )
    .iloc[0]
)

condition_mask = (
    primary_analysis_signatures["pert_id"].eq(
        highest_multiplicity_condition["pert_id"]
    )
    & primary_analysis_signatures["cell_iname"].eq(
        highest_multiplicity_condition["cell_iname"]
    )
    & primary_analysis_signatures["pert_dose"].eq(
        highest_multiplicity_condition["pert_dose"]
    )
    & primary_analysis_signatures["pert_time"].eq(
        highest_multiplicity_condition["pert_time"]
    )
)

target_sig_ids = set(
    primary_analysis_signatures.loc[
        condition_mask,
        "sig_id",
    ]
)

detailed_siginfo_columns = [
    "sig_id",
    "pert_id",
    "cell_iname",
    "pert_dose",
    "pert_time",
    "project_code",
    "bead_batch",
    "pert_mfc_id",
    "det_plates",
    "distil_ids",
    "nsample",
    "qc_pass",
    "is_hiq",
]

target_siginfo_chunks = []

for chunk in pd.read_csv(
    LINCS_SIGINFO_PATH,
    sep="\t",
    usecols=detailed_siginfo_columns,
    chunksize=250_000,
    low_memory=False,
):
    selected = chunk.loc[
        chunk["sig_id"].isin(target_sig_ids)
    ]

    if not selected.empty:
        target_siginfo_chunks.append(selected)

target_condition_siginfo = pd.concat(
    target_siginfo_chunks,
    ignore_index=True,
)

print(
    f"Target condition signatures: "
    f"{len(target_condition_siginfo):,}"
)
print(
    f"Unique projects: "
    f"{target_condition_siginfo['project_code'].nunique(dropna=True):,}"
)
print(
    f"Unique bead batches: "
    f"{target_condition_siginfo['bead_batch'].nunique(dropna=True):,}"
)
print(
    f"Unique pert_mfc_id: "
    f"{target_condition_siginfo['pert_mfc_id'].nunique(dropna=True):,}"
)

display(
    target_condition_siginfo[
        [
            "sig_id",
            "project_code",
            "bead_batch",
            "pert_mfc_id",
            "nsample",
            "is_hiq",
        ]
    ].head(20)
)

In [ ]:
# =============================================================================
# Inspect definitions of Level 5 replicate-traceability fields
# =============================================================================

replicate_traceability_fields = [
    "bead_batch",
    "pert_mfc_id",
    "det_plates",
    "distil_ids",
    "nsample",
]

replicate_traceability_definitions = (
    metadata_definitions["siginfo"]
    .loc[
        metadata_definitions["siginfo"]["Field Name"].isin(
            replicate_traceability_fields
        )
    ]
    .reset_index(drop=True)
)

replicate_traceability_definitions

In [ ]:
# =============================================================================
# Audit reuse of distilled replicate profiles across Level 5 signatures
# =============================================================================

target_distil_links = (
    target_condition_siginfo[
        [
            "sig_id",
            "distil_ids",
            "nsample",
        ]
    ]
    .assign(
        distil_id=lambda df: df["distil_ids"].str.split(
            "|",
            regex=False,
        )
    )
    .explode("distil_id")
)

target_distil_links["distil_id"] = (
    target_distil_links["distil_id"]
    .astype("string")
    .str.strip()
)

target_distil_links = target_distil_links.loc[
    target_distil_links["distil_id"].notna()
    & target_distil_links["distil_id"].ne("")
].copy()

links_per_signature = (
    target_distil_links
    .groupby("sig_id")
    .size()
    .rename("observed_nsample")
)

expected_nsample = (
    target_condition_siginfo
    .set_index("sig_id")["nsample"]
    .rename("expected_nsample")
)

nsample_check = pd.concat(
    [
        links_per_signature,
        expected_nsample,
    ],
    axis=1,
)

replicate_reuse = (
    target_distil_links
    .groupby("distil_id")["sig_id"]
    .nunique()
)

n_nsample_mismatch = (
    nsample_check["observed_nsample"]
    != nsample_check["expected_nsample"]
).sum()

n_reused_distil_ids = (
    replicate_reuse > 1
).sum()

max_signatures_per_distil_id = (
    replicate_reuse.max()
)

print(
    f"Total signature-to-replicate links: "
    f"{len(target_distil_links):,}"
)
print(
    f"Sum of nsample:                   "
    f"{target_condition_siginfo['nsample'].sum():,}"
)
print(
    f"Signatures with nsample mismatch: "
    f"{n_nsample_mismatch:,}"
)
print(
    f"Unique underlying distil_ids:     "
    f"{replicate_reuse.size:,}"
)
print(
    f"distil_ids reused across >1 sig:  "
    f"{n_reused_distil_ids:,}"
)
print(
    f"Maximum signatures per distil_id: "
    f"{max_signatures_per_distil_id:,}"
)

In [ ]:
# =============================================================================
# Load replicate traceability for the full primary signature universe
# =============================================================================

primary_signature_ids = set(
    primary_analysis_signatures["sig_id"]
)

primary_traceability_chunks = []

for chunk in pd.read_csv(
    LINCS_SIGINFO_PATH,
    sep="\t",
    usecols=[
        "sig_id",
        "distil_ids",
        "nsample",
    ],
    chunksize=250_000,
    low_memory=False,
):
    selected = chunk.loc[
        chunk["sig_id"].isin(primary_signature_ids)
    ]

    if not selected.empty:
        primary_traceability_chunks.append(selected)

primary_signature_traceability = pd.concat(
    primary_traceability_chunks,
    ignore_index=True,
)

print(
    f"Primary signatures expected: "
    f"{len(primary_signature_ids):,}"
)
print(
    f"Traceability rows recovered: "
    f"{len(primary_signature_traceability):,}"
)
print(
    f"Unique sig_id recovered:     "
    f"{primary_signature_traceability['sig_id'].nunique():,}"
)
print(
    f"Missing distil_ids:           "
    f"{primary_signature_traceability['distil_ids'].isna().sum():,}"
)

In [ ]:
# =============================================================================
# Audit global reuse of distilled replicate profiles
# =============================================================================

primary_distil_links = (
    primary_signature_traceability
    .assign(
        distil_id=lambda df: df["distil_ids"].str.split(
            "|",
            regex=False,
        )
    )
    .explode("distil_id")
)

primary_distil_links["distil_id"] = (
    primary_distil_links["distil_id"]
    .astype("string")
    .str.strip()
)

primary_distil_links = primary_distil_links.loc[
    primary_distil_links["distil_id"].notna()
    & primary_distil_links["distil_id"].ne("")
].copy()

links_per_signature = (
    primary_distil_links
    .groupby("sig_id")
    .size()
    .rename("observed_nsample")
)

expected_nsample = (
    primary_signature_traceability
    .set_index("sig_id")["nsample"]
    .rename("expected_nsample")
)

nsample_check = pd.concat(
    [
        links_per_signature,
        expected_nsample,
    ],
    axis=1,
)

replicate_reuse = (
    primary_distil_links
    .groupby("distil_id")["sig_id"]
    .nunique()
)

n_nsample_mismatch = (
    nsample_check["observed_nsample"]
    != nsample_check["expected_nsample"]
).sum()

n_reused_distil_ids = (
    replicate_reuse > 1
).sum()

max_signatures_per_distil_id = (
    replicate_reuse.max()
)

print(
    f"Total signature-to-replicate links: "
    f"{len(primary_distil_links):,}"
)
print(
    f"Sum of nsample:                   "
    f"{primary_signature_traceability['nsample'].sum():,}"
)
print(
    f"Signatures with nsample mismatch: "
    f"{n_nsample_mismatch:,}"
)
print(
    f"Unique underlying distil_ids:     "
    f"{replicate_reuse.size:,}"
)
print(
    f"distil_ids reused across >1 sig:  "
    f"{n_reused_distil_ids:,}"
)
print(
    f"Maximum signatures per distil_id: "
    f"{max_signatures_per_distil_id:,}"
)

## Frozen handling of Level 5 replicate structure

The LINCS 2020 Beta Level 5 `trt_cp` resource contains replicate-collapsed
signatures, but a nominal experimental condition
(`pert_id × cell line × dose × time`) may be represented by more than one
Level 5 signature.

Within the final primary universe:

- 362,036 Level 5 signatures map to 913,432 underlying `distil_id` profiles;
- the number of parsed `distil_id` values agrees exactly with `nsample`;
- no underlying `distil_id` is reused across more than one Level 5 signature.

Therefore, Level 5 signatures are treated as provider-defined,
replicate-collapsed experimental units based on non-overlapping underlying
profiles. However, multiple Level 5 signatures representing the same nominal
compound/cell/dose/time condition will not be treated as independent
compound-level evidence.

For Phase 7 connectivity analysis, connectivity will first be calculated at
the individual Level 5 signature level. Multiple signatures sharing the same
`pert_id × cell line × dose × time` condition will then be summarized
robustly at the condition level before aggregation across dose/time, cell
lines, lineages, and compounds.

No best-performing signature, dose, time point, cell line, or lineage will be
selected.

In [ ]:
# =============================================================================
# Characterize within-cell dose and time coverage
# =============================================================================

perturbagen_cell_condition_coverage = (
    condition_multiplicity
    .groupby(
        [
            "pert_id",
            "cell_iname",
        ],
        as_index=False,
    )
    .agg(
        n_conditions=("n_level5_signatures", "size"),
        n_level5_signatures=("n_level5_signatures", "sum"),
        n_doses=("pert_dose", "nunique"),
        n_times=("pert_time", "nunique"),
    )
)

print(
    f"Perturbagen × cell-line pairs: "
    f"{len(perturbagen_cell_condition_coverage):,}"
)

print("\nCoverage per perturbagen × cell-line pair:")
display(
    perturbagen_cell_condition_coverage[
        [
            "n_conditions",
            "n_level5_signatures",
            "n_doses",
            "n_times",
        ]
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

within_cell_structure = pd.DataFrame(
    {
        "category": [
            "single condition",
            "multiple conditions",
            "multiple doses",
            "multiple times",
            "multiple doses and times",
        ],
        "n_pairs": [
            perturbagen_cell_condition_coverage[
                "n_conditions"
            ].eq(1).sum(),
            perturbagen_cell_condition_coverage[
                "n_conditions"
            ].gt(1).sum(),
            perturbagen_cell_condition_coverage[
                "n_doses"
            ].gt(1).sum(),
            perturbagen_cell_condition_coverage[
                "n_times"
            ].gt(1).sum(),
            (
                perturbagen_cell_condition_coverage["n_doses"].gt(1)
                & perturbagen_cell_condition_coverage["n_times"].gt(1)
            ).sum(),
        ],
    }
)

within_cell_structure["pct_pairs"] = (
    100
    * within_cell_structure["n_pairs"]
    / len(perturbagen_cell_condition_coverage)
)

print("\nWithin-cell experimental structure:")
display(within_cell_structure)

## Frozen within-cell aggregation structure

The LINCS/CMap primary universe contains heterogeneous dose and time coverage
within individual perturbagen × cell-line pairs. Among 90,673 evaluable pairs,
67.4% contain multiple experimental conditions, 52.9% contain multiple doses,
and 23.1% contain multiple time points.

Phase 7 will therefore preserve the experimental hierarchy rather than pooling
all available signatures or selecting a favorable condition.

For each perturbagen and cell line, connectivity will be aggregated as follows:

1. individual Level 5 signatures are scored separately;
2. signatures sharing the same compound × cell line × dose × time condition
   are summarized by the median;
3. dose-level condition scores are summarized by the median within each time
   point;
4. time-point summaries are then summarized by the median to obtain one
   perturbagen × cell-line value.

This hierarchy prevents densely sampled dose series from receiving additional
weight merely because more conditions were measured. No best dose, time point,
or individual signature will be selected.

In [ ]:
# =============================================================================
# Audit compound metadata coverage and multiplicity
# =============================================================================

primary_compoundinfo = (
    lincs_compoundinfo
    .loc[
        lincs_compoundinfo["pert_id"].isin(
            final_primary_perturbagen_ids
        )
    ]
    .copy()
)

compoundinfo_pert_ids = set(
    primary_compoundinfo["pert_id"]
)

missing_compound_metadata = (
    final_primary_perturbagen_ids
    - compoundinfo_pert_ids
)

compound_row_multiplicity = (
    primary_compoundinfo
    .groupby("pert_id")
    .size()
    .rename("n_compoundinfo_rows")
)

print(
    f"Primary perturbagens:              "
    f"{len(final_primary_perturbagen_ids):,}"
)
print(
    f"Perturbagens represented in compoundinfo: "
    f"{len(compoundinfo_pert_ids):,}"
)
print(
    f"Missing from compoundinfo:         "
    f"{len(missing_compound_metadata):,}"
)

print("\ncompoundinfo rows per perturbagen:")
display(
    compound_row_multiplicity
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

print("\nMetadata completeness:")
display(
    pd.DataFrame(
        {
            "field": [
                "cmap_name",
                "target",
                "moa",
                "canonical_smiles",
                "inchi_key",
            ],
            "missing_rows": [
                primary_compoundinfo[column].isna().sum()
                for column in [
                    "cmap_name",
                    "target",
                    "moa",
                    "canonical_smiles",
                    "inchi_key",
                ]
            ],
            "non_missing_perturbagens": [
                primary_compoundinfo.loc[
                    primary_compoundinfo[column].notna(),
                    "pert_id",
                ].nunique()
                for column in [
                    "cmap_name",
                    "target",
                    "moa",
                    "canonical_smiles",
                    "inchi_key",
                ]
            ],
        }
    )
)

print(
    "\nPerturbagens with >1 compoundinfo row: "
    f"{(compound_row_multiplicity > 1).sum():,}"
)

In [ ]:
# =============================================================================
# Characterize compound annotation multiplicity
# =============================================================================

compound_annotation_structure = (
    primary_compoundinfo
    .groupby("pert_id", as_index=False)
    .agg(
        n_rows=("pert_id", "size"),
        n_cmap_names=("cmap_name", "nunique"),
        n_targets=("target", "nunique"),
        n_moa=("moa", "nunique"),
        n_smiles=("canonical_smiles", "nunique"),
        n_inchi_keys=("inchi_key", "nunique"),
    )
)

print(
    "Perturbagens with multiple cmap_name values: "
    f"{(compound_annotation_structure['n_cmap_names'] > 1).sum():,}"
)
print(
    "Perturbagens with multiple target annotations: "
    f"{(compound_annotation_structure['n_targets'] > 1).sum():,}"
)
print(
    "Perturbagens with multiple MoA annotations: "
    f"{(compound_annotation_structure['n_moa'] > 1).sum():,}"
)
print(
    "Perturbagens with multiple SMILES values: "
    f"{(compound_annotation_structure['n_smiles'] > 1).sum():,}"
)
print(
    "Perturbagens with multiple InChIKeys: "
    f"{(compound_annotation_structure['n_inchi_keys'] > 1).sum():,}"
)

print("\nHighest annotation multiplicity:")
display(
    compound_annotation_structure
    .sort_values(
        ["n_rows", "pert_id"],
        ascending=[False, True],
    )
    .head(10)
)

In [ ]:
# =============================================================================
# Inspect perturbagens with multiple CMap names
# =============================================================================

multi_name_pert_ids = set(
    compound_annotation_structure.loc[
        compound_annotation_structure["n_cmap_names"] > 1,
        "pert_id",
    ]
)

multi_name_compound_summary = (
    primary_compoundinfo
    .loc[
        primary_compoundinfo["pert_id"].isin(
            multi_name_pert_ids
        )
    ]
    .groupby("pert_id", as_index=False)
    .agg(
        n_rows=("pert_id", "size"),
        cmap_names=(
            "cmap_name",
            lambda values: " | ".join(
                sorted(
                    set(
                        values
                        .dropna()
                        .astype(str)
                    )
                )
            ),
        ),
        n_targets=("target", "nunique"),
        n_moa=("moa", "nunique"),
        n_smiles=("canonical_smiles", "nunique"),
        n_inchi_keys=("inchi_key", "nunique"),
    )
    .sort_values("pert_id")
    .reset_index(drop=True)
)

multi_name_compound_summary

In [ ]:
# =============================================================================
# Audit chemical-structure identifier reuse across perturbagen IDs
# =============================================================================

chemical_identity_records = []
shared_structure_examples = []

for identifier in [
    "inchi_key",
    "canonical_smiles",
]:
    identifier_pairs = (
        primary_compoundinfo[
            [
                "pert_id",
                identifier,
            ]
        ]
        .dropna()
        .drop_duplicates()
    )

    perturbagens_per_identifier = (
        identifier_pairs
        .groupby(identifier)["pert_id"]
        .nunique()
    )

    shared_identifiers = perturbagens_per_identifier.loc[
        perturbagens_per_identifier > 1
    ]

    pert_ids_with_shared_identifier = set(
        identifier_pairs.loc[
            identifier_pairs[identifier].isin(
                shared_identifiers.index
            ),
            "pert_id",
        ]
    )

    chemical_identity_records.append(
        {
            "identifier": identifier,
            "n_perturbagens_with_identifier": (
                identifier_pairs["pert_id"].nunique()
            ),
            "n_unique_identifier_values": (
                identifier_pairs[identifier].nunique()
            ),
            "n_identifier_values_shared_across_pert_ids": (
                len(shared_identifiers)
            ),
            "n_perturbagens_in_shared_identifier_groups": (
                len(pert_ids_with_shared_identifier)
            ),
            "max_perturbagens_per_identifier": (
                int(perturbagens_per_identifier.max())
                if not perturbagens_per_identifier.empty
                else 0
            ),
        }
    )

    if not shared_identifiers.empty:
        examples = (
            identifier_pairs
            .loc[
                identifier_pairs[identifier].isin(
                    shared_identifiers.index
                )
            ]
            .merge(
                shared_identifiers
                .rename("n_perturbagens")
                .reset_index(),
                on=identifier,
                how="left",
                validate="many_to_one",
            )
            .sort_values(
                ["n_perturbagens", identifier, "pert_id"],
                ascending=[False, True, True],
            )
            .head(20)
            .assign(identifier_type=identifier)
            .rename(columns={identifier: "identifier_value"})
        )

        shared_structure_examples.append(
            examples[
                [
                    "identifier_type",
                    "identifier_value",
                    "n_perturbagens",
                    "pert_id",
                ]
            ]
        )

chemical_identity_summary = pd.DataFrame(
    chemical_identity_records
)

display(chemical_identity_summary)

if shared_structure_examples:
    print("\nRepresentative cross-pert_id structure reuse:")
    display(
        pd.concat(
            shared_structure_examples,
            ignore_index=True,
        )
    )


## Frozen perturbagen identity and chemical annotation handling

`pert_id` is the authoritative **experimental perturbagen key** for the
Phase 7 LINCS/CMap analysis. It is not assumed to be a globally unique
chemical-structure identifier.

All perturbagens in the final primary universe must be represented in
`compoundinfo`. Compound metadata are not one-to-one with `pert_id`;
target and mechanism-of-action annotations may be many-to-many, and
`cmap_name` may contain synonym-level variation.

Chemical structure identifiers (`InChIKey` and canonical SMILES) are audited
both within and across `pert_id` values. Reuse of a structure identifier
across distinct perturbagen IDs is retained explicitly rather than collapsed
during the acquisition audit. Any downstream deduplication or structure-aware
sensitivity analysis must be prospectively specified before compound-level
prioritization.

Accordingly:

- `pert_id` defines the LINCS/CMap experimental perturbagen unit;
- `cmap_name` is descriptive metadata rather than an analytical key;
- target and mechanism-of-action annotations remain explicitly many-to-many;
- missing target or mechanism-of-action annotation is not interpreted as
  evidence of biological absence;
- chemical-structure identifiers remain available to detect duplicate or
  synonymous perturbagen representations; and
- compound annotation does not determine primary connectivity eligibility or
  ranking.

Mechanism-of-action aggregation in notebook 703 must preserve the underlying
many-to-many compound–target–MoA structure rather than forcing one perturbagen
into a single mechanism class.


## Frozen primary LINCS/CMap gene space

The primary Phase 7 perturbational analysis will use the **BING gene space**,
defined in this LINCS 2020 Beta release as the union of:

- 978 `landmark` genes, and
- 9,196 `best inferred` genes,

for a total of 10,174 BING genes.

This choice was made outcome-blind, before calculating any connectivity score,
based exclusively on representation fidelity of the three frozen Phase 4
consensus transcriptomic programs.

Exact gene-symbol mapping showed that each consensus program contains:

- 2,389 frozen Phase 4 genes;
- 1,769 genes represented anywhere in the Level 5 L1000 matrix;
- 1,495 genes represented in BING;
- only 90 genes represented in the landmark-only space.

Landmark-only representation retained only approximately 3–5% of the absolute
consensus-weight mass and 3–4% of squared-weight mass, with cosine fidelity of
approximately 0.17–0.21. It therefore provides a substantially truncated
representation of all three frozen programs.

BING retained approximately 64–69% of absolute weight mass and 64–72% of
squared-weight mass, with cosine fidelity of approximately 0.80–0.85. Both
signed arms were represented in all three programs, although
`CONSENSUS_TX_03` showed lower retention of its negative arm
(approximately 55% of its absolute negative weight mass), which will be
preserved as a representation limitation.

The complete 12,328-gene Level 5 space provides somewhat greater retention but
requires inclusion of the lower-priority `inferred` feature-space category.
It will therefore not define the primary representation.

Accordingly:

- **Primary gene space:** BING (`landmark` + `best inferred`);
- **Measured-only sensitivity:** `landmark`;
- the frozen Phase 4 consensus weights and orientations will not be refit,
  reweighted, or reoriented for LINCS/CMap;
- genes unavailable in a given LINCS gene space will be omitted from that
  representation rather than imputed or rescued using downstream results.

The Level 5 HDF5 numerical array is physically stored as signature × gene.
The persisted handoffs therefore use unambiguous `gctx_signature_index` and
`gctx_gene_index` fields; direct `h5py` access in downstream notebooks must
index the matrix as `matrix[gctx_signature_index, gctx_gene_index]`.

This decision concerns representation fidelity only and does not constitute
evidence of perturbational reversal or therapeutic activity.

In [ ]:
# =============================================================================
# Materialize frozen Phase 7 gene-space representations
# =============================================================================

PRIMARY_GENE_SPACES = {
    "bing": {
        "landmark",
        "best inferred",
    },
    "landmark": {
        "landmark",
    },
}

phase7_bing_weights = (
    consensus_weights_lincs
    .loc[
        consensus_weights_lincs["feature_space"].isin(
            PRIMARY_GENE_SPACES["bing"]
        )
    ]
    .copy()
    .sort_values(
        [
            "consensus_program_id",
            "gene_symbol",
        ]
    )
    .reset_index(drop=True)
)

phase7_landmark_weights = (
    consensus_weights_lincs
    .loc[
        consensus_weights_lincs["feature_space"].isin(
            PRIMARY_GENE_SPACES["landmark"]
        )
    ]
    .copy()
    .sort_values(
        [
            "consensus_program_id",
            "gene_symbol",
        ]
    )
    .reset_index(drop=True)
)

for space_name, weights_df in {
    "BING": phase7_bing_weights,
    "landmark": phase7_landmark_weights,
}.items():
    assert not weights_df["consensus_weight"].isna().any()
    assert not weights_df.duplicated(
        [
            "consensus_program_id",
            "gene_symbol",
        ]
    ).any()

    genes_per_program = (
        weights_df
        .groupby("consensus_program_id")["gene_symbol"]
        .nunique()
        .to_dict()
    )

    print(f"{space_name}:")
    print(f"  rows: {len(weights_df):,}")
    print(f"  genes per program: {genes_per_program}")

In [ ]:
# =============================================================================
# Build deterministic GCTX gene and signature handoff indices
# =============================================================================

gctx_gene_index_table = pd.DataFrame(
    {
        "gene_id": pd.to_numeric(
            pd.Series(gctx_gene_ids),
            errors="raise",
        ).astype("int64"),
        "gctx_gene_index": np.arange(
            len(gctx_gene_ids),
            dtype=int,
        ),
    }
)

gctx_signature_index_table = pd.DataFrame(
    {
        "sig_id": gctx_signature_ids,
        "gctx_signature_index": np.arange(
            len(gctx_signature_ids),
            dtype=int,
        ),
    }
)

phase7_bing_gene_handoff = (
    phase7_bing_weights
    .assign(
        gene_id=lambda df: pd.to_numeric(
            df["gene_id"],
            errors="raise",
        ).astype("int64")
    )
    .merge(
        gctx_gene_index_table,
        on="gene_id",
        how="left",
        validate="many_to_one",
    )
)

phase7_landmark_gene_handoff = (
    phase7_landmark_weights
    .assign(
        gene_id=lambda df: pd.to_numeric(
            df["gene_id"],
            errors="raise",
        ).astype("int64")
    )
    .merge(
        gctx_gene_index_table,
        on="gene_id",
        how="left",
        validate="many_to_one",
    )
)

phase7_signature_handoff = (
    primary_analysis_signatures
    .merge(
        gctx_signature_index_table,
        on="sig_id",
        how="left",
        validate="one_to_one",
    )
)

assert phase7_bing_gene_handoff["gctx_gene_index"].notna().all()
assert phase7_landmark_gene_handoff["gctx_gene_index"].notna().all()
assert phase7_signature_handoff["gctx_signature_index"].notna().all()

print(
    f"BING handoff rows:      "
    f"{len(phase7_bing_gene_handoff):,}"
)
print(
    f"Landmark handoff rows:  "
    f"{len(phase7_landmark_gene_handoff):,}"
)
print(
    f"Signature handoff rows: "
    f"{len(phase7_signature_handoff):,}"
)
print(
    f"Unique GCTX signatures in primary handoff: "
    f"{phase7_signature_handoff['gctx_signature_index'].nunique():,}"
)
print(
    "Direct h5py access uses the physical matrix order "
    "matrix[gctx_signature_index, gctx_gene_index]."
)


In [ ]:
# =============================================================================
# Persist LINCS/CMap audit and Phase 7 handoff artifacts
# =============================================================================

LINCS_INTERIM_DIR = Paths.perturbational
LINCS_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

LINCS_INPUT_INVENTORY_PATH = (
    LINCS_INTERIM_DIR
    / "110_lincs_raw_input_inventory.csv"
)

LINCS_REPRESENTATION_FIDELITY_PATH = (
    LINCS_INTERIM_DIR
    / "110_lincs_program_representation_fidelity.csv"
)

LINCS_BING_GENE_HANDOFF_PATH = (
    LINCS_INTERIM_DIR
    / "110_lincs_bing_consensus_gene_handoff.csv"
)

LINCS_LANDMARK_GENE_HANDOFF_PATH = (
    LINCS_INTERIM_DIR
    / "110_lincs_landmark_consensus_gene_handoff.csv"
)

LINCS_SIGNATURE_HANDOFF_PATH = (
    LINCS_INTERIM_DIR
    / "110_lincs_primary_signature_handoff.parquet"
)

LINCS_PERTURBAGEN_COVERAGE_PATH = (
    LINCS_INTERIM_DIR
    / "110_lincs_primary_perturbagen_coverage.csv"
)

LINCS_COMPOUND_ANNOTATION_PATH = (
    LINCS_INTERIM_DIR
    / "110_lincs_primary_compound_annotation_handoff.csv"
)

primary_compound_annotation_handoff = (
    primary_compoundinfo
    .loc[
        primary_compoundinfo["pert_id"].isin(
            final_primary_perturbagen_ids
        )
    ]
    .sort_values(
        [
            "pert_id",
            "cmap_name",
            "target",
            "moa",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

lincs_file_inventory.to_csv(
    LINCS_INPUT_INVENTORY_PATH,
    index=False,
)

representation_fidelity.to_csv(
    LINCS_REPRESENTATION_FIDELITY_PATH,
    index=False,
)

phase7_bing_gene_handoff.to_csv(
    LINCS_BING_GENE_HANDOFF_PATH,
    index=False,
)

phase7_landmark_gene_handoff.to_csv(
    LINCS_LANDMARK_GENE_HANDOFF_PATH,
    index=False,
)

phase7_signature_handoff.to_parquet(
    LINCS_SIGNATURE_HANDOFF_PATH,
    index=False,
)

final_primary_coverage.to_csv(
    LINCS_PERTURBAGEN_COVERAGE_PATH,
    index=False,
)

primary_compound_annotation_handoff.to_csv(
    LINCS_COMPOUND_ANNOTATION_PATH,
    index=False,
)

print("LINCS/CMap audit and handoff artifacts written: 7")
print(
    f"Directory: "
    f"{project_relative_path(LINCS_INTERIM_DIR)}"
)

In [ ]:
# =============================================================================
# Re-read, verify, and fingerprint persisted LINCS/CMap handoff artifacts
# =============================================================================

lincs_handoff_specs = {
    "phase1.110.raw_input_inventory": {
        "path": LINCS_INPUT_INVENTORY_PATH,
        "expected": lincs_file_inventory,
    },
    "phase1.110.program_representation_fidelity": {
        "path": LINCS_REPRESENTATION_FIDELITY_PATH,
        "expected": representation_fidelity,
    },
    "phase1.110.bing_gene_handoff": {
        "path": LINCS_BING_GENE_HANDOFF_PATH,
        "expected": phase7_bing_gene_handoff,
    },
    "phase1.110.landmark_gene_handoff": {
        "path": LINCS_LANDMARK_GENE_HANDOFF_PATH,
        "expected": phase7_landmark_gene_handoff,
    },
    "phase1.110.primary_signature_handoff": {
        "path": LINCS_SIGNATURE_HANDOFF_PATH,
        "expected": phase7_signature_handoff,
    },
    "phase1.110.primary_perturbagen_coverage": {
        "path": LINCS_PERTURBAGEN_COVERAGE_PATH,
        "expected": final_primary_coverage,
    },
    "phase1.110.primary_compound_annotation": {
        "path": LINCS_COMPOUND_ANNOTATION_PATH,
        "expected": primary_compound_annotation_handoff,
    },
}

lincs_handoff_records = []

for artifact_id, spec in lincs_handoff_specs.items():
    path = spec["path"]
    expected = spec["expected"]

    if not path.exists():
        raise FileNotFoundError(
            f"Expected handoff artifact was not written: {path}"
        )

    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(
            f"Invalid or empty handoff artifact: {path}"
        )

    if path.suffix == ".parquet":
        persisted = pd.read_parquet(path)
    elif path.suffix == ".csv":
        persisted = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(
            f"Unsupported handoff format for validation: {path}"
        )

    if persisted.shape != expected.shape:
        raise RuntimeError(
            f"Persisted shape mismatch for {artifact_id}: "
            f"{persisted.shape} != {expected.shape}"
        )

    if persisted.columns.tolist() != expected.columns.tolist():
        raise RuntimeError(
            f"Persisted column mismatch for {artifact_id}."
        )

    lincs_handoff_records.append(
        {
            "artifact_id": artifact_id,
            "path": project_relative_path(path),
            "rows": int(persisted.shape[0]),
            "columns": int(persisted.shape[1]),
            "size_bytes": int(path.stat().st_size),
            "sha256": calculate_sha256(path),
        }
    )

lincs_handoff_inventory = pd.DataFrame(
    lincs_handoff_records
)

print(
    f"LINCS/CMap handoff artifacts independently re-read and verified: "
    f"{len(lincs_handoff_inventory)}/"
    f"{len(lincs_handoff_specs)}"
)

display(lincs_handoff_inventory)


In [ ]:
# =============================================================================
# Build and validate proposed LINCS/CMap raw-data registry entry
# =============================================================================

LINCS_DOWNLOAD_PAGE_URL = (
    "https://clue.io/data/project/5fb7d248c1f885001161d6c8"
)

lincs_raw_file_specs = {
    "cellinfo_beta.txt": {
        "role": "cell_line_metadata",
        "status": "acquired_and_used",
    },
    "compoundinfo_beta.txt": {
        "role": "compound_metadata",
        "status": "acquired_and_used",
    },
    "geneinfo_beta.txt": {
        "role": "gene_metadata",
        "status": "acquired_and_used",
    },
    "instinfo_beta.txt": {
        "role": "instance_metadata",
        "status": "acquired_and_used",
    },
    "level5_beta_trt_cp_n720216x12328.gctx": {
        "role": "level5_compound_signatures",
        "status": "acquired_and_used",
    },
    "LINCS2020 Release Metadata Field Definitions.xlsx": {
        "role": "metadata_field_definitions",
        "status": "supporting",
    },
    "README.txt": {
        "role": "resource_readme",
        "status": "supporting",
    },
    "siginfo_beta.txt": {
        "role": "signature_metadata",
        "status": "acquired_and_used",
    },
}

lincs_inventory_by_filename = (
    lincs_file_inventory
    .assign(
        file_name=lambda df: (
            df["relative_path"]
            .str.replace("\\", "/", regex=False)
            .str.rsplit("/", n=1)
            .str[-1]
        )
    )
    .set_index("file_name")
)

lincs_registry_files = {}

for file_name, metadata in lincs_raw_file_specs.items():
    file_path = Paths.lincs / file_name

    if not file_path.exists() or not file_path.is_file():
        raise FileNotFoundError(
            f"Expected LINCS raw file not found: {file_path}"
        )

    if file_name not in lincs_inventory_by_filename.index:
        raise RuntimeError(
            f"Raw-file inventory entry missing for: {file_name}"
        )

    inventory_row = lincs_inventory_by_filename.loc[file_name]

    if int(file_path.stat().st_size) != int(inventory_row["size_bytes"]):
        raise RuntimeError(
            f"Raw-file size changed after initial audit: {file_name}"
        )

    lincs_registry_files[file_name] = {
        **metadata,
        "size_bytes": int(inventory_row["size_bytes"]),
        "sha256": str(inventory_row["sha256"]),
    }

lincs_registry_entry = {
    "source_database": "Connectivity Map (CMap) / LINCS",
    "provider": "Broad Institute",
    "release": "Expanded CMap LINCS Resource 2020 Beta",
    "provenance_mode": "file_managed",
    "canonical_dir": "data/raw/lincs",
    "files": lincs_registry_files,
    "download_page_url": LINCS_DOWNLOAD_PAGE_URL,
}

proposed_raw_data_registry = load_raw_data_registry()
proposed_raw_data_registry["lincs"] = lincs_registry_entry

validated_raw_data_registry = validate_raw_data_registry(
    proposed_raw_data_registry
)

validated_lincs_entry = validated_raw_data_registry["lincs"]

print("LINCS/CMap registry proposal validation: PASS")
print(
    "Release:",
    validated_lincs_entry["release"],
)
print(
    "Registered files:",
    len(validated_lincs_entry["files"]),
)
print(
    "Download page:",
    validated_lincs_entry["download_page_url"],
)


In [ ]:
# =============================================================================
# Persist and verify audited LINCS/CMap raw-data registry entry
# =============================================================================

with Paths.raw_data_registry.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    json.dump(
        validated_raw_data_registry,
        handle,
        indent=2,
        ensure_ascii=False,
    )
    handle.write("\n")

persisted_raw_data_registry = load_raw_data_registry(
    Paths.raw_data_registry
)

persisted_lincs_entry = persisted_raw_data_registry["lincs"]

if persisted_lincs_entry != validated_lincs_entry:
    raise RuntimeError(
        "Persisted LINCS registry entry does not match "
        "the validated proposal."
    )

if len(persisted_lincs_entry["files"]) != 8:
    raise RuntimeError(
        "Unexpected number of registered LINCS files."
    )

print("Persisted LINCS/CMap raw-data registry validation: PASS")
print(
    "Registry:",
    project_relative_path(Paths.raw_data_registry),
)
print(
    "Status:",
    persisted_lincs_entry["release"],
)
print(
    "Registered files:",
    len(persisted_lincs_entry["files"]),
)

In [ ]:
# =============================================================================
# Register frozen notebook 110 LINCS/CMap artifacts
# =============================================================================

with Paths.artifact_registry.open(
    "r",
    encoding="utf-8",
) as handle:
    artifact_registry = json.load(handle)

artifacts = artifact_registry["artifacts"]

producer = {
    "type": "notebook",
    "path": (
        "notebooks/phase1_data_acquisition_and_auditing/"
        "110_lincs_cmap_acquisition_and_audit.ipynb"
    ),
}

handoff_inventory_by_id = (
    lincs_handoff_inventory
    .set_index("artifact_id")
    .to_dict(orient="index")
)


def build_registered_artifact(
    artifact_id,
    artifact_role,
    inputs,
):
    """Build one frozen artifact-registry entry from audited fingerprints."""
    spec = handoff_inventory_by_id[artifact_id]

    return {
        "path": spec["path"],
        "phase": 1,
        "status": "frozen",
        "artifact_role": artifact_role,
        "producer": producer,
        "shape": [
            int(spec["rows"]),
            int(spec["columns"]),
        ],
        "size_bytes": int(spec["size_bytes"]),
        "sha256": spec["sha256"],
        "inputs": inputs,
    }


lincs_raw_refs = [
    {
        "type": "raw_registry",
        "ref": f"/lincs/files/{file_name}",
    }
    for file_name in lincs_raw_file_specs
]

artifacts["phase1.110.raw_input_inventory"] = (
    build_registered_artifact(
        "phase1.110.raw_input_inventory",
        "metadata",
        lincs_raw_refs,
    )
)

artifacts["phase1.110.program_representation_fidelity"] = (
    build_registered_artifact(
        "phase1.110.program_representation_fidelity",
        "metadata",
        [
            {
                "type": "artifact",
                "artifact_id": (
                    "phase4.401.consensus_transcriptomic_gene_weights"
                ),
            },
            {
                "type": "raw_registry",
                "ref": "/lincs/files/geneinfo_beta.txt",
            },
        ],
    )
)

for artifact_id, artifact_role in [
    ("phase1.110.bing_gene_handoff", "handoff"),
    ("phase1.110.landmark_gene_handoff", "handoff"),
]:
    artifacts[artifact_id] = build_registered_artifact(
        artifact_id,
        artifact_role,
        [
            {
                "type": "artifact",
                "artifact_id": (
                    "phase4.401.consensus_transcriptomic_gene_weights"
                ),
            },
            {
                "type": "raw_registry",
                "ref": "/lincs/files/geneinfo_beta.txt",
            },
            {
                "type": "raw_registry",
                "ref": (
                    "/lincs/files/"
                    "level5_beta_trt_cp_n720216x12328.gctx"
                ),
            },
        ],
    )

artifacts["phase1.110.primary_signature_handoff"] = (
    build_registered_artifact(
        "phase1.110.primary_signature_handoff",
        "handoff",
        [
            {
                "type": "raw_registry",
                "ref": "/lincs/files/siginfo_beta.txt",
            },
            {
                "type": "raw_registry",
                "ref": "/lincs/files/cellinfo_beta.txt",
            },
            {
                "type": "raw_registry",
                "ref": (
                    "/lincs/files/"
                    "level5_beta_trt_cp_n720216x12328.gctx"
                ),
            },
        ],
    )
)

artifacts["phase1.110.primary_perturbagen_coverage"] = (
    build_registered_artifact(
        "phase1.110.primary_perturbagen_coverage",
        "metadata",
        [
            {
                "type": "artifact",
                "artifact_id": (
                    "phase1.110.primary_signature_handoff"
                ),
            },
        ],
    )
)

artifacts["phase1.110.primary_compound_annotation"] = (
    build_registered_artifact(
        "phase1.110.primary_compound_annotation",
        "handoff",
        [
            {
                "type": "artifact",
                "artifact_id": (
                    "phase1.110.primary_perturbagen_coverage"
                ),
            },
            {
                "type": "raw_registry",
                "ref": "/lincs/files/compoundinfo_beta.txt",
            },
        ],
    )
)

with Paths.artifact_registry.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    json.dump(
        artifact_registry,
        handle,
        indent=2,
        ensure_ascii=False,
    )
    handle.write("\n")

print("Registered frozen notebook 110 artifacts: 7")
print(
    "Registry:",
    project_relative_path(Paths.artifact_registry),
)

In [ ]:
# =============================================================================
# Validate frozen notebook 110 registry publication
# =============================================================================

with Paths.artifact_registry.open(
    "r",
    encoding="utf-8",
) as handle:
    persisted_artifact_registry = json.load(handle)

persisted_raw_data_registry = load_raw_data_registry(
    Paths.raw_data_registry
)

registered_artifacts = persisted_artifact_registry["artifacts"]

expected_artifacts = {
    row.artifact_id: {
        "path": row.path,
        "shape": [
            int(row.rows),
            int(row.columns),
        ],
        "size_bytes": int(row.size_bytes),
        "sha256": row.sha256,
    }
    for row in lincs_handoff_inventory.itertuples(index=False)
}

artifact_validation_records = []

for artifact_id, expected in expected_artifacts.items():
    entry = registered_artifacts.get(artifact_id)

    artifact_validation_records.append(
        {
            "artifact_id": artifact_id,
            "exists": entry is not None,
            "path_matches": (
                entry is not None
                and entry["path"] == expected["path"]
            ),
            "shape_matches": (
                entry is not None
                and entry["shape"] == expected["shape"]
            ),
            "size_matches": (
                entry is not None
                and entry["size_bytes"] == expected["size_bytes"]
            ),
            "sha256_matches": (
                entry is not None
                and entry["sha256"] == expected["sha256"]
            ),
            "status_frozen": (
                entry is not None
                and entry["status"] == "frozen"
            ),
        }
    )

artifact_validation = pd.DataFrame(
    artifact_validation_records
)


def registry_ref_exists(registry, ref):
    """Return whether a slash-delimited registry reference resolves."""
    node = registry

    for part in ref.strip("/").split("/"):
        if not isinstance(node, dict) or part not in node:
            return False

        node = node[part]

    return True


unresolved_artifact_inputs = []
unresolved_raw_inputs = []

for artifact_id in expected_artifacts:
    entry = registered_artifacts[artifact_id]

    for input_spec in entry.get("inputs", []):
        if input_spec["type"] == "artifact":
            upstream_id = input_spec["artifact_id"]

            if upstream_id not in registered_artifacts:
                unresolved_artifact_inputs.append(
                    (artifact_id, upstream_id)
                )

        elif input_spec["type"] == "raw_registry":
            raw_ref = input_spec["ref"]

            if not registry_ref_exists(
                persisted_raw_data_registry,
                raw_ref,
            ):
                unresolved_raw_inputs.append(
                    (artifact_id, raw_ref)
                )

validation_columns = [
    "exists",
    "path_matches",
    "shape_matches",
    "size_matches",
    "sha256_matches",
    "status_frozen",
]

if not artifact_validation[validation_columns].all().all():
    raise RuntimeError(
        "At least one notebook 110 artifact registry entry "
        "failed validation."
    )

if unresolved_artifact_inputs:
    raise RuntimeError(
        "Unresolved artifact dependencies detected: "
        f"{unresolved_artifact_inputs}"
    )

if unresolved_raw_inputs:
    raise RuntimeError(
        "Unresolved raw-registry dependencies detected: "
        f"{unresolved_raw_inputs}"
    )

print("Notebook 110 registry publication validation: PASS")
print(
    f"Frozen artifacts validated: "
    f"{len(expected_artifacts)}/{len(expected_artifacts)}"
)
print(
    f"Unresolved artifact inputs: "
    f"{len(unresolved_artifact_inputs)}"
)
print(
    f"Unresolved raw-data inputs: "
    f"{len(unresolved_raw_inputs)}"
)

display(artifact_validation)

## Notebook completion

Notebook 110 completes the acquisition, integrity audit, metadata
characterization, and downstream handoff of the Expanded CMap LINCS Resource
2020 Beta required for Phase 7.

### Audited source resource

Eight static LINCS/CMap source files were audited and registered, including the
Level 5 `trt_cp` GCTX matrix and its associated signature, instance, cell-line,
gene, compound, field-definition, and release metadata.

The Level 5 matrix contains:

- 12,328 genes;
- 720,216 compound signatures;
- exact correspondence between all 720,216 GCTX signature IDs and the
  `trt_cp` subset of `siginfo`.

Raw-file byte identity is frozen through locally calculated SHA-256
fingerprints in `config/raw_data_registry.json`.

### Primary Phase 7 signature universe

The primary perturbational universe is restricted prospectively to:

- Level 5 `trt_cp` signatures;
- provider `qc_pass == 1`;
- tumor cell lines with an explicitly annotated, non-`unknown` lineage;
- perturbagens represented in at least 5 distinct tumor cell lines and at
  least 4 distinct annotated lineages;
- complete dose and time metadata required to define the nominal experimental
  condition.

After applying these criteria and re-evaluating coverage following the
complete-condition requirement, the frozen primary handoff contains:

- 362,036 Level 5 signatures;
- 7,914 perturbagens.

`is_hiq` is retained for a stricter technical sensitivity rather than as a
primary eligibility criterion. Its sensitivity subset must independently
satisfy the same 5-cell-line / 4-lineage gate after all primary technical
requirements are applied.

Normal, pooled, and tumor cell lines with unknown lineage are excluded from
the primary lineage-balanced universe and may be used only as explicitly
defined contextual sensitivity analyses.

### Replicate and condition structure

Level 5 signatures are provider-defined replicate-collapsed units.

Across the primary universe, 362,036 Level 5 signatures map to 913,432
underlying `distil_id` profiles. Parsed `distil_id` counts agree exactly with
`nsample`, and no underlying profile is reused across multiple retained Level
5 signatures.

Multiple Level 5 signatures can nevertheless represent the same nominal
compound × cell line × dose × time condition. Phase 7 will therefore preserve
the experimental hierarchy rather than treating these signatures as
independent compound-level evidence.

Within each perturbagen × cell-line context, aggregation will follow:

1. Level 5 signature;
2. nominal dose × time condition;
3. dose within time;
4. time within cell line.

Robust median summaries will be used at each collapse. No best-performing
signature, dose, time point, cell line, or lineage will be selected.

### Frozen consensus-program representation

Phase 4 consensus transcriptomic programs remain unchanged.

The primary LINCS representation uses the BING gene space
(`landmark` + `best inferred`). Each of the three frozen consensus programs
maps to 1,495 BING genes.

The landmark-only sensitivity representation contains 90 genes per program
and showed substantially lower outcome-blind representation fidelity.

The BING representation preserves substantially more of the frozen Phase 4
weight structure, although `CONSENSUS_TX_03` retains a lower fraction of its
negative signed arm than the other program arms. This limitation must remain
explicit in downstream interpretation.

No Phase 4 weight, sign, orientation, or program membership was refit or
modified using LINCS/CMap data. Persisted GCTX handoffs use explicit
`gctx_signature_index` and `gctx_gene_index` fields; direct `h5py` access
must follow the physical signature × gene matrix order.

### Perturbagen identity and annotation

`pert_id` is the authoritative LINCS/CMap experimental perturbagen key; it is
not assumed to define a globally unique chemical structure.

All 7,914 primary perturbagens are represented in `compoundinfo`. Target and
mechanism-of-action annotations are explicitly many-to-many and incomplete;
missing annotation is not interpreted as biological absence. InChIKey and
canonical-SMILES reuse across distinct `pert_id` values is audited explicitly
and retained rather than silently collapsed.

`cmap_name` is descriptive metadata rather than an analytical key.
Mechanism-of-action aggregation in notebook 703 must preserve the underlying
many-to-many compound–target–MoA structure.

### Frozen handoff artifacts

Seven deterministic artifacts have been written under
`data/interim/perturbational/` and registered as frozen in
`config/artifact_registry.json`:

- `phase1.110.raw_input_inventory`
- `phase1.110.program_representation_fidelity`
- `phase1.110.bing_gene_handoff`
- `phase1.110.landmark_gene_handoff`
- `phase1.110.primary_signature_handoff`
- `phase1.110.primary_perturbagen_coverage`
- `phase1.110.primary_compound_annotation`

All registered paths, shapes, file sizes, SHA-256 fingerprints, artifact
dependencies, and raw-data references were independently validated after
publication.

### Analytical boundary

Notebook 110 does not calculate connectivity, inspect inverse-signature
results, prioritize compounds or mechanisms, or use Phase 5/6 evidence to
define the perturbational universe.

Inverse perturbational associations generated downstream are computational
perturbational hypotheses. They do not establish therapeutic reversal,
treatment efficacy, causal mechanism, or clinical relevance.

The audited and frozen notebook-110 handoff is ready for prospective Phase 7
analysis.